# CLIP-ViT + Transformer — Dynamic Logits + Contrastive + Attention-Pooling trên Flickr30k

Bản chuyển từ `coco2014-clip-vit-dynlogits-contrastive-attnpool.ipynb` sang **Flickr30k (Karpathy split)**, kèm cell **đánh giá trên tập test sau khi train xong**.

Giữ nguyên kiến trúc và huấn luyện: Dynamic Logits tại đầu ra decoder (chặn mềm theo xác suất, warmup 10 epoch), contrastive đa dương tính (SupCon, 3 negative: 2 đổi object + 1 hoán vị trí kiểu NegCLIP), attention-pooling, label smoothing 0,1, cắt gradient 1,0, AMP.

## Khác so với bản COCO

1. **Dữ liệu**: `dataset_flickr30k.json` (train 29.000 / val 1.014 / test 1.000 ảnh, 5 câu mỗi ảnh), không có restval. Đường dẫn tự dò dưới `/kaggle/input`; cần gắn Karpathy split Flickr30k và ảnh Flickr30k.
2. **Nhãn object**: Flickr30k không có COCO instances, nên object "có trong ảnh" được rút từ 5 caption của ảnh (`build_caption_objects`, 80 lớp COCO). Vật thể thấy được nhưng không caption nào nhắc sẽ bị coi là vắng mặt, nên Dynamic Logits và negative phạt nhầm nhiều hơn bản COCO. Đây là chỗ cần xem kỹ khi đọc kết quả.
3. **Số epoch**: `EPOCHS` = 30 (mỗi epoch 1.512 bước, ít hơn COCO khoảng 3,9 lần).
4. **Đánh giá**: cell cuối chạy tự động sau khi train (greedy và beam search, BLEU/METEOR/ROUGE-L/CIDEr, CHAIRs/CHAIRi). Dynamic Logits ở đây chỉ dùng lúc train; lúc đánh giá không chặn logits (không có bằng chứng YOLO hay câu truy hồi).

Tên checkpoint/từ điển có tiền tố `flickr30k_karpathy_dynlogits_contrastive_attnpool_`.


In [ ]:
import os
import math
import re
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from collections import Counter
from transformers import CLIPVisionModel, get_linear_schedule_with_warmup
import json  # Thêm json để xử lý COCO annotation và lưu vocab

# ==========================================
# CẤU HÌNH VÀ SIÊU THAM SỐ (TRAIN FROM SCRATCH)
# ==========================================
SEED = 42

def seed_everything(seed=SEED):
    """Cố định RNG để các lần train có thể tái lập."""
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = True  # fixed IMAGE_SIZE/BATCH_SIZE -> an toan bat autotune cuDNN, tang toc

def seed_worker(worker_id):
    """Seed Python/NumPy trong từng DataLoader worker."""
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Đường dẫn Flickr30k trên Kaggle. None = tự dò dưới /kaggle/input, nên chỉ cần gắn (Add Input) 2 dataset:
#   1) Karpathy split của Flickr30k (file dataset_flickr30k.json: train 29.000 / val 1.014 / test 1.000 ảnh, 5 câu mỗi ảnh),
#   2) ảnh Flickr30k (thư mục chứa các file .jpg, vd 1000092795.jpg).
# Có thể điền tay nếu dò sai.
KARPATHY_JSON_FILE = None
FLICKR_IMG_DIR = None
# Dự phòng khi không có file Karpathy: captions.txt (cột image,caption) hoặc results.csv (image_name| comment_number| comment).
# Khi đó ảnh được chia ngẫu nhiên (seed cố định) 29.000 / 1.014 / 1.000, KHÔNG trùng split Karpathy trong các bài báo.
FLICKR_CAPTIONS_FILE = None
# Flickr30k Karpathy không có restval: tập huấn luyện chỉ gồm "train".
TRAIN_SPLITS = ("train",)

# Đường dẫn lưu trữ - đặt tên riêng "scratch" để không đè lên checkpoint của bản fine-tune cũ
OUTPUT_DIR = "/kaggle/working"
os.makedirs(OUTPUT_DIR, exist_ok=True)

VOCAB_FILE = os.path.join(OUTPUT_DIR, "flickr30k_karpathy_dynlogits_contrastive_attnpool_vocab.json")
CHECKPOINT_FILE = os.path.join(OUTPUT_DIR, "flickr30k_karpathy_dynlogits_contrastive_attnpool_last_checkpoint.pth")
BEST_MODEL_FILE = os.path.join(OUTPUT_DIR, "flickr30k_karpathy_dynlogits_contrastive_attnpool_best_model.pth")

# Ablation MỚI (thêm contrastive loss + projection head mới) -> train từ đầu, KHÔNG kế thừa
# checkpoint của cấu hình dynamic-logits+fixloss (kiến trúc đã đổi, load lại sẽ lệch shape).
# Nếu kernel bị Kaggle ngắt giữa chừng, đổi RESUME_TRAINING=True và trỏ RESUME_CHECKPOINT_FILE
# về đúng last_checkpoint mà CHÍNH bản contrastive này đã lưu (không phải checkpoint bản khác).
RESUME_TRAINING = False
RESUME_CHECKPOINT_FILE = CHECKPOINT_FILE
FINE_TUNE_RESET_TRAINING_STATE = False
# Train tiếp khi lịch LR cũ đã chạy hết (LR ~ 0, vd tăng EPOCHS 10 -> 19): đặt LR đỉnh nhỏ (vd 2e-5)
# để dựng lịch warmup + giảm tuyến tính MỚI cho các epoch còn lại. None = giữ đúng lịch cũ.
RESUME_PEAK_LR = None

# Hallucination-aware training: Flickr30k không có COCO instances, nên nhãn object của ảnh rút từ chính 5 caption
# (xem build_caption_objects). Không dùng YOLO.
USE_HALL_LOSS = False  # tắt theo yêu cầu: chỉ dùng Dynamic Logits (decoder intercept) + Contrastive
USE_CAPTION_OBJECT_LABELS = True
USE_YOLO_HALL_LOSS = False
LAMBDA_HALL = 0.01
HALL_LAMBDA_WARMUP_EPOCHS = 3
HALL_LOSS_MIN_STEP_RATIO = 0.3
HALL_DEBUG_BATCHES = 2

# --- Contrastive loss ảnh-caption (xem research/contrastive-loss-for-caption-hallucination.md) ---
# Positive = caption đúng (teacher forcing); negative = caption đúng nhưng bị thay 1 object đang
# CÓ trong ảnh bằng 1 object KHÔNG có trong ảnh (mục 5.1 tài liệu) -> tái dùng đúng
# COCO_OBJECT_SYNONYMS/object_token_ids đã có sẵn cho HalluLoss, không cần nguồn dữ liệu mới.
USE_CONTRASTIVE_LOSS = True
LAMBDA_CONTRASTIVE = 0.01  # nâng tu 0.005 (mục 8 tai lieu: "neu CHAIR khong giam du, thu lambda_con=0.01")
NUM_NEGATIVES = 3  # 2 hallucination-style (doi object) + 1 structural-style (hoan vi tri,
                   # kieu NegCLIP) - can cu: Wu et al. 2022, NegCLIP (Yuksekgonul et al. 2023).
NUM_NEGATIVES_STRUCTURAL = 1  # trong NUM_NEGATIVES, so luong la kieu NegCLIP: hoan vi tri 2
                              # object DANG CO trong caption, GIU NGUYEN tu vung (bag-of-words)
                              # -> ep hoc dung binding/thu tu, khac kieu hallucination
SKIP_UNSWAPPABLE_NEGATIVES = True  # True (ban sua): negative khong doi/hoan duoc (caption khong co token object de thay)
                                   # KHONG chen them tu vao cuoi cau nua (cau bi chen ngu phap sai -> model hoc loi tat "cau la"
                                   # thay vi nhin anh) ma bi LOAI khoi mau so InfoNCE. False = hanh vi cu cua E010-E015
                                   # (chen object vao cuoi cau, hoac giu nguyen thanh ban sao cua positive).
EXPAND_OBJECT_PLURALS = True  # True (ban sua): them dang so nhieu (beds, dogs, men...) co trong vocab vao tap token cua moi
                              # object -- dung cho Dynamic Logits, hall loss va tao negative. False = chi dang so it (E010-E015).
K_POS_PER_IMAGE = 2  # I006 diem 1+4 (SupCon - Khosla et al. 2020): 1 anchor (dung cho CE
                     # nhu cu) + (K-1) caption THAT khac cua CUNG anh, chi dung cho contrastive.
                     # Caption dung cua anh KHAC trong batch tu dong la in-batch negative,
                     # khong can ham sinh rieng (xem multi_positive_infonce_loss).
CONTRASTIVE_TAU = 0.07
CONTRASTIVE_WARMUP_EPOCHS = 3  # gate(epoch) = min(1, epoch/warmup), giống cách HALL_LAMBDA_WARMUP_EPOCHS
CONTRASTIVE_PROJ_DIM = 256  # chiều chung để chiếu image/text embedding trước khi tính cosine similarity
POOLING_MODE = "attention"  # I006 diem 2 (Nhom B): "mean" (nhu N010/N011a/N011b, mac dinh
                            # cua kien truc cu) hoac "attention" (AttentionPool moi trong
                            # notebook nay). De o day (khong phai True/False rieng) de 1 bien
                            # duy nhat dieu khien ca project_image va project_text, tien so
                            # sanh ablation neu can chay lai voi POOLING_MODE="mean" tren
                            # CHINH notebook nay (kien truc AttentionPool luon duoc khoi tao,
                            # chi khong duoc GOI toi khi POOLING_MODE="mean").
CONTRASTIVE_ATTN_POOL_HEADS = 4  # so head cua nn.MultiheadAttention trong AttentionPool,
                                 # theo dung de xuat cua giang vien (khong dang ke so voi
                                 # encoder/decoder chinh).
USE_AMP = True  # mixed precision (fp16 autocast + GradScaler) cho vong train - chi tang toc, KHONG dung cho vong val de giu metric val nguyen fp32 nhu cac run truoc

# --- Dừng an toàn trước giới hạn 12 giờ/phiên Kaggle ---
# Hết mỗi epoch (sau khi đã lưu last_checkpoint): nếu giờ đã chạy + 1,1 x thời gian epoch vừa rồi vượt ngân sách thì dừng
# hẳn, phiên kết thúc bình thường và output được lưu. Phiên sau resume bằng RESUME_TRAINING=True. None = không giới hạn.
TRAIN_TIME_BUDGET_HOURS = 11.5
NOTEBOOK_START_TIME = __import__("time").time()

IMAGE_SIZE = (224, 224)
SEQ_LENGTH = 25
MIN_WORD_FREQ = 5
EMBED_DIM = 512
# --- Giữ NGUYÊN kiến trúc decoder giống hệt bản baseline/fine-tune (FF_DIM=512, NUM_LAYERS=2) ---
# Để so sánh baseline vs bản train-from-scratch (hallucination loss + decoder intercept + BLEU
# checkpoint selection) là fair, KHÔNG đổi kiến trúc ở đây - nếu đổi cùng lúc, kết quả tốt hơn sẽ
# không biết là do decoder to hơn hay do các cải tiến hallucination đang muốn đánh giá. Muốn thử
# kiến trúc lớn hơn thì đổi trực tiếp 2 số này, coi đó là 1 thí nghiệm RIÊNG, KHÔNG gộp chung
# với thí nghiệm so sánh baseline.
FF_DIM = 512
NUM_LAYERS = 2
NUM_HEADS = 8
BATCH_SIZE = 96  # 32 -> 96: GPU nhanh hơn ~1,5 lần/mẫu, đỉnh bộ nhớ ~2,2 GB (đo trên RTX 3050 với AMP),
                 # Kaggle 16 GB dư sức. Lưu ý: số negative trong batch của contrastive tăng 3 lần và số bước/epoch giảm 3 lần.

# Dùng lại đúng LR baseline-from-scratch của bạn (1e-4) để so sánh fair - không đoán 1 giá trị
# mới (vd 4e-4) khi chưa có bằng chứng nó hợp với đúng kiến trúc/data pipeline này. Fine-tune
# (tiếp tục train từ checkpoint có sẵn) mới cần LR nhỏ hơn hẳn (vd 1e-5), không áp dụng ở đây.
EPOCHS = 30  # 1.512 bước/epoch (29.000 ảnh x 5 câu / 96); bản COCO chạy 40 epoch x 5.904 bước
LEARNING_RATE = 1e-4

# --- Cải tiến chống overfit / hallucination khi train from scratch ---
LABEL_SMOOTHING = 0.1  # giảm overconfidence -> giảm hallucination
GRAD_CLIP_NORM = 1.0   # ổn định gradient khi train decoder từ đầu

# --- Dynamic intercept ngay tại đầu ra decoder TRONG LÚC TRAIN ---
# Bản cũ dùng penalty tĩnh cho mọi timestep nên dễ làm méo CE loss và có thể khiến CHAIR tăng.
# Bản dynamic này mềm hơn:
# 1) ramp-up penalty theo epoch để các epoch đầu học ngôn ngữ/grounding ổn trước,
# 2) chỉ phạt mạnh ở timestep mà model đang gán xác suất cao cho nhóm object vắng mặt,
# 3) không phạt token target đúng khi tính CE, tránh xung đột nếu COCO instances thiếu object.
TRAIN_DECODER_INTERCEPT = True
TRAIN_INTERCEPT_MODE = "dynamic"
TRAIN_INTERCEPT_MAX_PENALTY = 0.5
TRAIN_INTERCEPT_WARMUP_EPOCHS = 10
TRAIN_INTERCEPT_PROB_THRESHOLD = 0.08
TRAIN_INTERCEPT_TARGET_PROTECT = True

# --- Cờ tổng hợp: có cần dựng object_token_ids/COCO-instance-objects hay không? ---
# 3 thành phần dùng CHUNG hạ tầng grounding này: HalluLoss (phạt loss), Dynamic Decoder
# Intercept (hạ logits) và Contrastive loss (tạo negative caption). USE_HALL_LOSS=False KHÔNG
# được phép tắt luôn hạ tầng này nếu Dynamic Intercept hoặc Contrastive vẫn cần nó - nếu không,
# apply_decoder_object_intercept/build_negative_captions sẽ luôn no-op dù bật cờ tương ứng.
# USE_HALL_LOSS chỉ còn quyết định lambda_hall có được cộng vào loss hay không (xem get_hall_lambda
# ở dưới, đã đổi thành current_lambda_hall = 0.0 khi USE_HALL_LOSS=False).
USE_OBJECT_GROUNDING = USE_HALL_LOSS or TRAIN_DECODER_INTERCEPT or USE_CONTRASTIVE_LOSS

# --- Theo dõi / early-stop bằng metric caption thật (BLEU-4) thay vì chỉ val_loss ---
# val_loss thấp không đồng nghĩa caption tốt hơn: dùng BLEU-4 chuẩn COCO caption (pycocoevalcap,
# KHÔNG dùng nltk vì nltk BLEU vốn thiết kế cho dịch máy, không khớp cách đánh giá image
# captioning trong các paper/benchmark COCO) trên 1 subset ảnh val mỗi epoch (greedy decode)
# để chọn best model theo chất lượng caption thực tế.
EARLY_STOP_METRIC = "loss"  # "bleu4" | "loss"
VAL_METRIC_MAX_SAMPLES = 1000

PAD_IDX = 0
START_IDX = 1
END_IDX = 2
UNK_IDX = 3

# ==========================================
# TIỀN XỬ LÝ DỮ LIỆU
# ==========================================
def custom_standardization(text):
    text = text.lower()
    strip_chars = r"""!"#$%&'()*+,-./:;=?@[\]^_`{|}~1234567890"""
    text = re.sub(f"[{re.escape(strip_chars)}]", "", text)
    return text.strip()

# Train: Có Augmentation (giảm RandomRotation so với bản fine-tune: 36 -> 15 độ, vì xoay mạnh
# làm lệch phân bố ảnh so với lúc CLIP pretrain và không cập nhật lại caption theo hướng xoay
# -> nhiễu label, đặc biệt hại khi train from scratch).
train_transforms = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.CenterCrop(IMAGE_SIZE),
    T.RandomHorizontalFlip(),
    T.RandomRotation(degrees=15),
    T.ColorJitter(contrast=0.3),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711])
])

# Validation: KHÔNG Augmentation
val_transforms = T.Compose([
    T.Resize(IMAGE_SIZE),
    T.CenterCrop(IMAGE_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.48145466, 0.4578275, 0.40821073],
                std=[0.26862954, 0.26130258, 0.27577711])
])

class FlickrDataset(Dataset):
    def __init__(self, image_paths, captions, vocab, transforms=None, image_objects=None, k_pos_per_image=1):
        self.image_paths = image_paths
        self.captions = captions
        self.vocab = vocab
        self.transforms = transforms
        self.image_objects = image_objects or {}
        self.k_pos_per_image = max(k_pos_per_image, 1)
        # I006 diem 1 (SupCon - Khosla et al. 2020): gom TAT CA caption THAT cung 1 anh (vd 5
        # caption Karpathy) de lay them (k_pos_per_image - 1) positive cho contrastive loss - CE
        # van chi dung dung 1 caption/idx nhu truoc, KHONG doi.
        self.image_to_captions = {}
        for path, cap in zip(image_paths, captions):
            self.image_to_captions.setdefault(path, []).append(cap)

    def __len__(self):
        return len(self.image_paths)

    def _tokenize(self, caption):
        tokens = [self.vocab.get(word, UNK_IDX) for word in caption.split()]
        tokens = [START_IDX] + tokens[:SEQ_LENGTH - 2] + [END_IDX]
        padded_tokens = tokens + [PAD_IDX] * (SEQ_LENGTH - len(tokens))
        return torch.tensor(padded_tokens, dtype=torch.long)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        caption = self.captions[idx]
        gt_objects_text = "|".join(sorted(self.image_objects.get(os.path.basename(img_path), set())))

        try:
            image = Image.open(img_path).convert("RGB")
        except Exception as e:
            # Fallback nếu ảnh bị lỗi (Rất hiếm nhưng có thể xảy ra ở bộ dữ liệu lớn)
            image = Image.new("RGB", IMAGE_SIZE, (0, 0, 0))

        if self.transforms:
            image = self.transforms(image)

        target_tensor = self._tokenize(caption)

        num_extra = self.k_pos_per_image - 1
        if num_extra > 0:
            # Ưu tiên caption KHÁC với anchor để tránh positive trùng hệt; nếu ảnh chỉ có 1
            # caption (hiếm), fallback lấy lại chính nó (không crash, chỉ giảm đa dạng).
            candidates = [c for c in self.image_to_captions.get(img_path, []) if c != caption]
            if not candidates:
                candidates = self.image_to_captions.get(img_path, [caption])
            if len(candidates) >= num_extra:
                extra_captions = random.sample(candidates, num_extra)
            else:
                extra_captions = [random.choice(candidates) for _ in range(num_extra)]
            extra_pos_tensor = torch.stack([self._tokenize(c) for c in extra_captions], dim=0)
        else:
            extra_pos_tensor = torch.zeros((0, SEQ_LENGTH), dtype=torch.long)

        return image, target_tensor, img_path, gt_objects_text, extra_pos_tensor

# ==========================================
# DỮ LIỆU FLICKR30K (Karpathy split)
# ==========================================
_FLICKR_META = None   # list[(filename, split, [câu gốc])]
SPLIT_SOURCE = None


def _find_first(root, predicate):
    for dirpath, _, files in os.walk(root):
        for name in files:
            if predicate(name):
                return os.path.join(dirpath, name)
    return None


def _load_flickr_meta():
    """Đọc danh sách (filename, split, câu gốc) một lần: ưu tiên dataset_flickr30k.json, nếu không có thì dùng file caption + chia ngẫu nhiên."""
    global _FLICKR_META, SPLIT_SOURCE, KARPATHY_JSON_FILE, FLICKR_IMG_DIR, FLICKR_CAPTIONS_FILE
    if _FLICKR_META is not None:
        return _FLICKR_META

    if KARPATHY_JSON_FILE is None and FLICKR_CAPTIONS_FILE is None:
        KARPATHY_JSON_FILE = _find_first("/kaggle/input", lambda n: n == "dataset_flickr30k.json")
        if KARPATHY_JSON_FILE is None:
            FLICKR_CAPTIONS_FILE = (_find_first("/kaggle/input", lambda n: n == "captions.txt")
                                    or _find_first("/kaggle/input", lambda n: n == "results.csv"))
    meta = []
    if KARPATHY_JSON_FILE:
        with open(KARPATHY_JSON_FILE, "r", encoding="utf-8") as f:
            data = json.load(f)
        for img in data.get("images", []):
            sents = [s["raw"] if "raw" in s else " ".join(s.get("tokens", [])) for s in img.get("sentences", [])]
            if img.get("filename") and sents:
                meta.append((img["filename"], img.get("split"), sents))
        SPLIT_SOURCE = f"Karpathy split ({KARPATHY_JSON_FILE})"
    elif FLICKR_CAPTIONS_FILE:
        by_image = {}
        with open(FLICKR_CAPTIONS_FILE, "r", encoding="utf-8") as f:
            header = f.readline()
            sep = "|" if "|" in header else ","
            for line in f:
                parts = line.rstrip("\n").split(sep, 2 if sep == "|" else 1)
                if len(parts) < 2:
                    continue
                name, cap = parts[0].strip(), parts[-1].strip()
                by_image.setdefault(name, []).append(cap)
        names = sorted(by_image)
        random.Random(SEED).shuffle(names)
        for i, name in enumerate(names):
            split = "train" if i < 29000 else "val" if i < 29000 + 1014 else "test" if i < 29000 + 1014 + 1000 else None
            if split:
                meta.append((name, split, by_image[name]))
        SPLIT_SOURCE = f"CHIA NGẪU NHIÊN từ {FLICKR_CAPTIONS_FILE} (không phải split Karpathy)"
    else:
        raise RuntimeError("Không tìm thấy dataset_flickr30k.json hay captions.txt/results.csv dưới /kaggle/input. "
                           "Hãy gắn dataset Flickr30k hoặc điền KARPATHY_JSON_FILE / FLICKR_CAPTIONS_FILE.")

    if FLICKR_IMG_DIR is None:
        probe = meta[0][0]
        hit = _find_first("/kaggle/input", lambda n: n == probe)
        if hit is None:
            raise RuntimeError(f"Không thấy ảnh {probe} dưới /kaggle/input. Hãy gắn dataset ảnh Flickr30k hoặc điền FLICKR_IMG_DIR.")
        FLICKR_IMG_DIR = os.path.dirname(hit)
    _FLICKR_META = meta
    return meta


def load_flickr_data(split="train"):
    """Trả về (image_paths, captions) theo cặp ảnh-caption; split là một tên hoặc tuple tên."""
    splits = {split} if isinstance(split, str) else set(split)
    image_paths, captions = [], []
    missing = 0
    for filename, sp, sents in _load_flickr_meta():
        if sp not in splits:
            continue
        path = os.path.join(FLICKR_IMG_DIR, filename)
        if not os.path.exists(path):
            missing += 1
            continue
        for raw in sents:
            caption = custom_standardization(raw)
            if caption:
                image_paths.append(path)
                captions.append(caption)
    print(f"-> Flickr30k {sorted(splits)}: {len(set(image_paths))} ảnh, {len(captions)} cặp ảnh-caption.")
    if missing:
        print(f"-> Bỏ qua {missing} ảnh vì không tìm thấy file ảnh.")
    return image_paths, captions


def build_or_load_vocab(captions, min_freq=5, vocab_file=VOCAB_FILE):
    """Hàm tạo Vocabulary mới hoặc tải từ file nếu đã tồn tại"""
    if os.path.exists(vocab_file):
        print(f"-> Tìm thấy file Vocab. Đang tải Vocabulary từ {vocab_file}...")
        with open(vocab_file, 'r', encoding='utf-8') as f:
            vocab = json.load(f)
        print(f"-> Kích thước Vocabulary đã tải: {len(vocab)}")
        return vocab

    print(f"-> Không tìm thấy file {vocab_file}. Đang xây dựng Vocabulary mới...")
    all_words = []
    for cap in captions:
        all_words.extend(cap.split())

    word_counts = Counter(all_words)
    valid_words = [word for word, count in word_counts.items() if count >= min_freq]

    vocab = {"<pad>": PAD_IDX, "<start>": START_IDX, "<end>": END_IDX, "<unk>": UNK_IDX}
    for idx, word in enumerate(valid_words):
        vocab[word] = idx + 4

    # Lưu lại Vocab để tái sử dụng
    with open(vocab_file, 'w', encoding='utf-8') as f:
        json.dump(vocab, f, ensure_ascii=False)

    print(f"-> Kích thước Vocabulary (từ xuất hiện >= {min_freq} lần): {len(vocab)}")
    print(f"-> Đã lưu Vocabulary tại: {vocab_file}")
    return vocab

In [ ]:
# ==========================================
# KIẾN TRÚC MÔ HÌNH
# ==========================================

# ==========================================
# HALLUCINATION-AWARE LOSS DỰA TRÊN COCO INSTANCES (GROUND TRUTH)
# ==========================================
COCO_OBJECT_SYNONYMS = {
    "person": ["person", "people", "man", "woman", "boy", "girl", "child", "children"],
    "bicycle": ["bicycle", "bike"], "car": ["car", "auto", "automobile"],
    "motorcycle": ["motorcycle", "motorbike"], "airplane": ["airplane", "plane", "aircraft"],
    "bus": ["bus"], "train": ["train"], "truck": ["truck"], "boat": ["boat"],
    "traffic light": ["traffic", "light"], "fire hydrant": ["hydrant"], "stop sign": ["stop", "sign"],
    "parking meter": ["meter"], "bench": ["bench"], "bird": ["bird"], "cat": ["cat"], "dog": ["dog"],
    "horse": ["horse"], "sheep": ["sheep"], "cow": ["cow"], "elephant": ["elephant"], "bear": ["bear"],
    "zebra": ["zebra"], "giraffe": ["giraffe"], "backpack": ["backpack"], "umbrella": ["umbrella"],
    "handbag": ["handbag", "bag", "purse"], "tie": ["tie"], "suitcase": ["suitcase", "luggage"],
    "frisbee": ["frisbee"], "skis": ["skis"], "snowboard": ["snowboard"], "sports ball": ["ball"],
    "kite": ["kite"], "baseball bat": ["bat"], "baseball glove": ["glove"], "skateboard": ["skateboard"],
    "surfboard": ["surfboard"], "tennis racket": ["racket"], "bottle": ["bottle"], "wine glass": ["glass"],
    "cup": ["cup"], "fork": ["fork"], "knife": ["knife"], "spoon": ["spoon"], "bowl": ["bowl"],
    "banana": ["banana"], "apple": ["apple"], "sandwich": ["sandwich"], "orange": ["orange"],
    "broccoli": ["broccoli"], "carrot": ["carrot"], "hot dog": ["hotdog"], "pizza": ["pizza"],
    "donut": ["donut", "doughnut"], "cake": ["cake"], "chair": ["chair"], "couch": ["couch", "sofa"],
    "potted plant": ["plant"], "bed": ["bed"], "dining table": ["table"], "toilet": ["toilet"],
    "tv": ["tv", "television"], "laptop": ["laptop"], "mouse": ["mouse"], "remote": ["remote"],
    "keyboard": ["keyboard"], "cell phone": ["phone", "cellphone"], "microwave": ["microwave"],
    "oven": ["oven"], "toaster": ["toaster"], "sink": ["sink"], "refrigerator": ["refrigerator", "fridge"],
    "book": ["book"], "clock": ["clock"], "vase": ["vase"], "scissors": ["scissors"],
    "teddy bear": ["teddy"], "hair drier": ["drier", "dryer"], "toothbrush": ["toothbrush"],
}

# Dạng số nhiều KHÔNG thêm vào bảng object vì nghĩa khác với object COCO:
# "glasses" trong caption thường là kính đeo mắt, không phải wine glass.
_PLURAL_EXCLUDE = {"glasses"}
# Dạng viết sai có trong vocab: vẫn dùng để NHẬN DIỆN/phạt token nhưng KHÔNG chọn làm từ thay thế khi tạo negative
# (câu kiểu "next to a mans" sai ngữ pháp, model nhận ra "câu lạ" mà không cần nhìn ảnh).
_REPLACEMENT_TYPOS = {"mans", "womans", "childs", "childrens", "peoples", "persons", "knifes", "mouses", "buss", "sheeps"}
_BASE_PLURAL_WORDS = {"people", "children"}  # dạng số nhiều đã có sẵn trong COCO_OBJECT_SYNONYMS
OBJECT_PLURAL_IDS = set()       # id các token số nhiều (gán trong build_object_token_ids khi EXPAND_OBJECT_PLURALS)
REPLACEMENT_EXCLUDE_IDS = set()  # id các token viết sai, không dùng làm từ thay thế
_PLURAL_IRREGULAR = {"man": ["men"], "woman": ["women"], "child": ["children"], "person": ["people", "persons"],
                      "mouse": ["mice"], "knife": ["knives"]}


def _plural_forms(word):
    """Các dạng số nhiều có thể của 1 từ đơn (cụm nhiều từ bỏ qua). Chỉ dạng nào CÓ trong vocab mới được dùng."""
    if " " in word:
        return []
    forms = list(_PLURAL_IRREGULAR.get(word, []))
    forms += [word + "s", word + "es"]
    if word.endswith("y") and len(word) > 1 and word[-2] not in "aeiou":
        forms.append(word[:-1] + "ies")
    if word.endswith("f"):
        forms.append(word[:-1] + "ves")
    if word.endswith("fe"):
        forms.append(word[:-2] + "ves")
    return forms


# category -> danh sách từ (số ít + số nhiều nếu EXPAND_OBJECT_PLURALS), được gán trong build_object_token_ids.
OBJECT_WORDS = {}


def build_object_token_ids(vocab):
    global OBJECT_WORDS, OBJECT_PLURAL_IDS, REPLACEMENT_EXCLUDE_IDS
    added_plurals = set()
    base_words = {w for words in COCO_OBJECT_SYNONYMS.values() for w in words}
    taken = set(base_words)  # không để dạng số nhiều "cướp" từ gốc của category khác hoặc của category đã lấy trước
    object_token_ids = {}
    OBJECT_WORDS = {}
    for obj_name, synonyms in COCO_OBJECT_SYNONYMS.items():
        words = list(synonyms)
        if EXPAND_OBJECT_PLURALS:
            for w in synonyms:
                for p in _plural_forms(w):
                    if p in vocab and p not in taken and p not in _PLURAL_EXCLUDE:
                        words.append(p)
                        taken.add(p)
                        added_plurals.add(p)
        OBJECT_WORDS[obj_name] = words
        ids = sorted({vocab[word] for word in words if word in vocab})
        if ids:
            object_token_ids[obj_name] = ids
    if EXPAND_OBJECT_PLURALS:
        OBJECT_PLURAL_IDS = {vocab[w] for w in (added_plurals | _BASE_PLURAL_WORDS) if w in vocab}
        REPLACEMENT_EXCLUDE_IDS = {vocab[w] for w in _REPLACEMENT_TYPOS if w in vocab}
    else:
        OBJECT_PLURAL_IDS, REPLACEMENT_EXCLUDE_IDS = set(), set()
    print(f"-> Hall loss theo dõi {len(object_token_ids)} object có token trong vocabulary.")
    return object_token_ids


def build_caption_objects(image_paths, captions):
    """Flickr30k không có COCO instances nên không có nhãn vật thể do người vẽ. Thay bằng nhãn rút từ chính 5 caption của ảnh:
    object (trong 80 lớp COCO) được coi là CÓ trong ảnh nếu ít nhất 1 caption của ảnh nhắc tới (qua OBJECT_WORDS, gồm số nhiều).
    Hệ quả cần nhớ: vật thể thấy được nhưng không caption nào nhắc sẽ bị coi là vắng mặt, nên Dynamic Logits và negative sẽ phạt
    nhầm nhiều hơn so với nhãn COCO instances. Khóa là tên file, giống bản COCO."""
    word_to_category = {}
    for category, words in OBJECT_WORDS.items():
        for w in words:
            word_to_category.setdefault(w, category)
    image_objects = {}
    for path, cap in zip(image_paths, captions):
        found = image_objects.setdefault(os.path.basename(path), set())
        for w in cap.split():
            category = word_to_category.get(w)
            if category is not None:
                found.add(category)
    print(f"-> Đã rút nhãn object từ caption cho {len(image_objects)} ảnh "
          f"(trung bình {sum(len(v) for v in image_objects.values()) / max(len(image_objects), 1):.2f} object/ảnh).")
    return image_objects

HALL_DEBUG_STATE = {"calls": 0}

def hallucination_object_loss(
    outputs,
    image_paths,
    gt_objects_batch,
    object_token_ids,
    token_mask=None,
    min_step_index=0,
):
    """Object-level unlikelihood loss đã làm mềm để phối hợp với dynamic logits.

    - Dùng mean trên nhóm token absent thay vì sum để ảnh có nhiều object vắng mặt
      không bị phạt mạnh chỉ vì số lượng token cần xét lớn hơn.
    - Chỉ phạt từ min_step_index trở đi để tránh nhiễu ở đầu câu, khi decoder còn
      ít ngữ cảnh ngôn ngữ.
    """
    if not object_token_ids:
        return outputs.new_tensor(0.0)

    probs = outputs.softmax(dim=-1)
    seq_len = outputs.size(1)
    if min_step_index > 0:
        step_positions = torch.arange(seq_len, device=outputs.device)
        step_position_mask = (step_positions >= min_step_index).float()
    else:
        step_position_mask = None

    losses = []
    num_with_gt_objects = 0
    num_with_present_objects = 0
    num_with_absent_tokens = 0
    for batch_index, image_path in enumerate(image_paths):
        gt_text = gt_objects_batch[batch_index] if gt_objects_batch is not None else ""
        gt_objects = set(gt_text.split("|")) if gt_text else set()
        if gt_objects:
            present_objects = gt_objects
            num_with_gt_objects += 1
            num_with_present_objects += 1
        else:
            continue

        absent_token_ids = []
        for obj_name, token_ids in object_token_ids.items():
            if obj_name not in present_objects:
                absent_token_ids.extend(token_ids)

        if not absent_token_ids:
            continue
        num_with_absent_tokens += 1

        absent_ids = torch.tensor(sorted(set(absent_token_ids)), device=outputs.device, dtype=torch.long)
        absent_probs = probs[batch_index].index_select(dim=-1, index=absent_ids).clamp(max=1.0 - 1e-6)
        step_penalty = -torch.log1p(-absent_probs).mean(dim=-1)

        if token_mask is not None:
            valid_steps = token_mask[batch_index].float()
        else:
            valid_steps = outputs.new_ones(step_penalty.shape)

        if step_position_mask is not None:
            valid_steps = valid_steps * step_position_mask

        valid_steps_sum = valid_steps.sum()
        if valid_steps_sum.item() <= 0:
            continue

        losses.append((step_penalty * valid_steps).sum() / valid_steps_sum.clamp_min(1.0))

    if not losses:
        if HALL_DEBUG_STATE["calls"] < HALL_DEBUG_BATCHES:
            print(
                f"[HallDebug] batch_size={len(image_paths)} | object_groups={len(object_token_ids)} | "
                f"gt_images={num_with_gt_objects} | present_images={num_with_present_objects} | "
                f"penalized_images={num_with_absent_tokens} | hall_ul=0.00000000"
            )
            HALL_DEBUG_STATE["calls"] += 1
        return outputs.new_tensor(0.0)

    if HALL_DEBUG_STATE["calls"] < HALL_DEBUG_BATCHES:
        debug_value = torch.stack(losses).mean().detach().item()
        print(
            f"[HallDebug] batch_size={len(image_paths)} | object_groups={len(object_token_ids)} | "
            f"gt_images={num_with_gt_objects} | present_images={num_with_present_objects} | "
            f"penalized_images={num_with_absent_tokens} | hall_ul={debug_value:.8f}"
        )
        HALL_DEBUG_STATE["calls"] += 1
    return torch.stack(losses).mean()


def get_hall_lambda(epoch, max_lambda=LAMBDA_HALL, warmup_epochs=HALL_LAMBDA_WARMUP_EPOCHS):
    if warmup_epochs <= 0:
        return max_lambda
    progress = min(1.0, float(epoch + 1) / float(warmup_epochs))
    return max_lambda * progress


# ==========================================
# CONTRASTIVE LOSS ẢNH-CAPTION (research/contrastive-loss-for-caption-hallucination.md)
# ==========================================
def build_object_id_to_name(object_token_ids):
    """Đảo ngược object_token_ids (obj_name -> [token_id,...]) đã có sẵn cho HalluLoss thành
    token_id -> obj_name, dùng để dò object-token trong 1 caption khi tạo negative."""
    id_to_name = {}
    for obj_name, token_ids in object_token_ids.items():
        for token_id in token_ids:
            id_to_name[token_id] = obj_name
    return id_to_name


def build_object_frequency_weights(captions, object_token_ids):
    """Đếm tần suất THẬT của mỗi category (qua toàn bộ synonym) trong ground-truth captions
    gốc (không phải liệt kê đều 80 category như trước). Căn cứ: Rohrbach et al. 2018 -
    hallucination thật tương quan với tần suất xuất hiện trong dữ liệu (vd "dining table" bị
    hallucinate nhiều nhất vì xuất hiện rất thường xuyên trong caption) - xem
    scratchpad_audit_do_kho_negative.html. Dùng làm trọng số sampling category thay thay vì
    random đều trên 80 category, giúp negative "khó" và thực tế hơn.

    Trả về dict category -> count (>= 1, cộng 1 Laplace smoothing để category hiếm vẫn có
    xác suất được chọn, không bị weight = 0).
    """
    word_to_category = {}
    for category in object_token_ids.keys():
        for word in (OBJECT_WORDS.get(category) or COCO_OBJECT_SYNONYMS.get(category, [])):
            word_to_category[word] = category

    weights = {name: 1 for name in object_token_ids.keys()}
    for caption in captions:
        seen_categories = set()
        for word in caption.split():
            category = word_to_category.get(word)
            if category is not None:
                seen_categories.add(category)
        for category in seen_categories:
            weights[category] += 1
    return weights


def _weighted_sample_without_replacement(names, weights, k):
    """Lấy k phần tử KHÔNG lặp từ names, theo trọng số weights (thuật toán
    Efraimidis-Spirakis: key = U^(1/w), lấy k key lớn nhất - tương đương weighted sampling
    không hoàn lại). weights=None hoặc rỗng -> uniform (random.sample thuờng)."""
    if not names:
        return []
    if not weights:
        return random.sample(names, min(k, len(names)))
    keyed = [
        (random.random() ** (1.0 / max(weights.get(name, 1), 1e-6)), name)
        for name in names
    ]
    keyed.sort(key=lambda pair: pair[0], reverse=True)
    return [name for _, name in keyed[:k]]


def _pick_replacement_token(token_ids, original_token):
    """Chọn token thay thế CÙNG SỐ (ít/nhiều) với token bị thay để câu không sai hòa hợp số (vd "a dogs"),
    và không chọn dạng viết sai. Chỉ bật khi EXPAND_OBJECT_PLURALS; tắt thì y hệt bản cũ (random.choice)."""
    if not EXPAND_OBJECT_PLURALS or not OBJECT_PLURAL_IDS:
        return random.choice(token_ids)
    pool = [t for t in token_ids if t not in REPLACEMENT_EXCLUDE_IDS]
    if original_token is not None:
        same_number = [t for t in pool if (t in OBJECT_PLURAL_IDS) == (original_token in OBJECT_PLURAL_IDS)]
        pool = same_number or pool
    return random.choice(pool or token_ids)


def build_negative_captions(
    captions, gt_objects_batch, object_token_ids, id_to_obj_name,
    num_negatives=1, num_structural=0, object_freq_weights=None, return_valid=False,
):
    """Tạo num_negatives negative caption/positive cho contrastive loss, gồm 2 kiểu (mục 5.1/5.2
    tài liệu nghiên cứu + CAN_CU_HOC_THUAT.html + scratchpad_audit_do_kho_negative.html):

    Kiểu 1 - "hallucination" (num_negatives - num_structural negative đầu): thay 1 object ĐANG CÓ
    thật trong ảnh bằng 1 object KHÔNG có trong ảnh. Category thay thế được chọn theo
    object_freq_weights (tần suất THẬT trong ground-truth captions, xem build_object_frequency_weights)
    thay vì random đều trên 80 category - negative thực tế/khó hơn, mô phỏng đúng thiên
    lệch hallucination thật (Rohrbach et al. 2018). Từ thay thế cũng được chọn ngẫu nhiên
    trong TẤT CẢ synonym của category (không chỉ lấy cố định synonym đầu tiên).

    Nếu caption không chứa token object nào đã có (swap_pos):
      - SKIP_UNSWAPPABLE_NEGATIVES = True (bản sửa, mặc định): KHÔNG tạo negative cho ca này. Câu được
        giữ nguyên trong tensor nhưng `negative_valid` = False để multi_positive_infonce_loss loại nó
        khỏi mẫu số. Lý do: chèn một danh từ vào cuối câu đã hoàn chỉnh cho ra câu ngữ pháp sai, model
        chỉ cần nhận ra câu "lạ" là phân biệt được, không cần nhìn ảnh (lối tắt).
      - SKIP_UNSWAPPABLE_NEGATIVES = False (hành vi cũ của E010-E015): fallback CHÈN THẬT (dịch chuỗi
        sang phải 1 ô, không ghi đè) object vắng mặt vào trước <end>/<pad> đầu tiên, CHỈ khi còn ít
        nhất 1 ô <pad> dư; nếu không đủ điều kiện thì giữ nguyên caption (negative == positive).
    (Trước đó nữa code GHI ĐÈ lên token cuối cùng bất kể từ loại gì -> phá ngữ pháp câu.)

    Kiểu 2 - "structural" (num_structural negative cuối, kiểu NegCLIP - Yuksekgonul et al. 2023):
    hoán đổi VỊ TRÍ của 2 object khác nhau ĐANG CÙNG CÓ trong caption, GIữ NGUYÊN toàn bộ từ
    vựng (bag-of-words không đổi). Nếu không đủ 2 object để hoán vị, giữ nguyên caption
    ở negative đó (tương tự fallback trên).

    Trả về tensor (B, num_negatives, L); nếu return_valid=True trả về (tensor, negative_valid) với
    negative_valid (B, num_negatives) bool = negative KHÁC positive và dùng được trong InfoNCE.
    """
    batch_size = captions.size(0)
    negative_captions = captions.unsqueeze(1).repeat(1, num_negatives, 1).clone()
    all_object_names = list(object_token_ids.keys())
    num_hallucination = max(num_negatives - num_structural, 0)

    for batch_index in range(batch_size):
        gt_text = gt_objects_batch[batch_index] if gt_objects_batch is not None else ""
        present_objects = set(gt_text.split("|")) if gt_text else set()
        seq = captions[batch_index].tolist()

        # --- Kiểu 1: hallucination (đổi object, ưu tiên category thường gặp trong ground-truth captions) ---
        if num_hallucination > 0:
            absent_names = [name for name in all_object_names if name not in present_objects]
            if absent_names:
                if len(absent_names) >= num_hallucination:
                    chosen_names = _weighted_sample_without_replacement(
                        absent_names, object_freq_weights, num_hallucination
                    )
                else:
                    pool_weights = (
                        [object_freq_weights.get(name, 1) for name in absent_names]
                        if object_freq_weights else None
                    )
                    chosen_names = random.choices(absent_names, weights=pool_weights, k=num_hallucination)

                swap_pos = next(
                    (pos for pos, tok in enumerate(seq) if id_to_obj_name.get(tok) in present_objects),
                    None,
                )
                can_insert = False
                insert_pos = None
                if swap_pos is None:
                    # Fallback: không có object-token nào để thay -> thử CHÈN (không ghi đè)
                    # object vắng mặt trước <end>/<pad> đầu tiên. Chỉ an toàn khi token cuối
                    # cùng của chuỗi là <pad> (có ô dư để "hy sinh" khi dịch phải 1 bước),
                    # tránh ghi đè lên từ bất kể (có thể không phải danh từ) như trước đây.
                    insert_pos = next((pos for pos, tok in enumerate(seq) if tok in (END_IDX, PAD_IDX)), None)
                    can_insert = (not SKIP_UNSWAPPABLE_NEGATIVES) and insert_pos is not None \
                        and insert_pos > 1 and seq[-1] == PAD_IDX

                for neg_index, replacement_name in enumerate(chosen_names):
                    replacement_id = _pick_replacement_token(
                        object_token_ids[replacement_name], seq[swap_pos] if swap_pos is not None else None
                    )
                    if swap_pos is not None:
                        negative_captions[batch_index, neg_index, swap_pos] = replacement_id
                    elif can_insert:
                        # Chèn thật: dịch seq[insert_pos:-1] sang phải 1 ô rồi mới ghi token mới
                        # vào insert_pos - không mất/ghi đè bất kỳ từ gốc nào (chỉ "hy sinh" 1 ô
                        # <pad> thừa ở cuối chuỗi).
                        negative_captions[batch_index, neg_index, insert_pos + 1:] = \
                            negative_captions[batch_index, neg_index, insert_pos:-1].clone()
                        negative_captions[batch_index, neg_index, insert_pos] = replacement_id
                    # Nếu không swap được và không thể chèn an toàn, giữ nguyên (negative ==
                    # positive, loss ~ ln(2) hằng số, không phá ngữ pháp).

        # --- Kiểu 2: structural (hoán vị trí, giữ nguyên từ vựng - kiểu NegCLIP) ---
        if num_structural > 0:
            object_positions = [pos for pos, tok in enumerate(seq) if id_to_obj_name.get(tok) in present_objects]
            distinct_pairs = [
                (a, b) for i, a in enumerate(object_positions) for b in object_positions[i + 1:]
                if seq[a] != seq[b]
            ]
            chosen_pairs = random.sample(distinct_pairs, min(num_structural, len(distinct_pairs))) if distinct_pairs else []

            for k, (pos_a, pos_b) in enumerate(chosen_pairs):
                slot = num_hallucination + k
                negative_captions[batch_index, slot, pos_a] = seq[pos_b]
                negative_captions[batch_index, slot, pos_b] = seq[pos_a]
            # Các slot structural còn lại (nếu không đủ cặp object khác nhau để hoán vị) giữ
            # nguyên (negative == positive, loss ~ ln(2) hằng số, không crash).

    if return_valid:
        if SKIP_UNSWAPPABLE_NEGATIVES:
            negative_valid = (negative_captions != captions.unsqueeze(1)).any(dim=-1)
        else:
            negative_valid = torch.ones(batch_size, num_negatives, dtype=torch.bool, device=captions.device)
        return negative_captions, negative_valid
    return negative_captions


def get_contrastive_lambda(epoch, max_lambda=LAMBDA_CONTRASTIVE, warmup_epochs=CONTRASTIVE_WARMUP_EPOCHS):
    """gate(epoch) * lambda_con, cùng công thức warmup với get_hall_lambda (mục 6.3 tài liệu)."""
    if warmup_epochs <= 0:
        return max_lambda
    progress = min(1.0, float(epoch + 1) / float(warmup_epochs))
    return max_lambda * progress


def multi_positive_infonce_loss(image_embed, text_embed, num_positives, tau=CONTRASTIVE_TAU, negative_valid=None):
    """SupCon-style multi-positive InfoNCE (Khosla et al. 2020, arXiv:2004.11362) + in-batch
    negative (I006 diem 1+4). Tong quat hoa cong thuc InfoNCE 1-positive cu (N009/N010) sang
    K positive/anh: vẫn cho kết quả giống hệt khi num_positives=1 và batch_size=1 (khong co
    in-batch negative).

    text_embed: (B, K+N, D) - K embedding positive THẬT (đầu, gồm anchor CE + extra positive
    cùng ảnh) rồi N embedding negative tổng hợp (cuối, từ build_negative_captions) - xếp
    theo đúng thứ tự này ở nơi gọi hàm.

    A(i) = positive của MọI ảnh (kể cả ảnh khác trong batch - in-batch negative, miễn phí,
    không cần hàm sinh riêng) + negative tổng hợp CủA CHÍNH ảnh i. Negative tổng hợp của
    ảnh KHÁC bị LOẠI khỏi A(i) (đúng công thức I006, vì negative sinh cho ảnh khác không
    chắc là negative hợp lệ cho ảnh này).

    negative_valid (B, N) bool, tùy chọn: False = negative đó (của ảnh b) không dùng được (không đổi/hoán được
    nên bị bỏ) -> loại khỏi mẫu số. Positive luôn hợp lệ.
    """
    batch_size, total_slots, dim = text_embed.shape
    flat_text = text_embed.reshape(batch_size * total_slots, dim)
    sim = (image_embed @ flat_text.t()) / tau
    sim = sim.view(batch_size, batch_size, total_slots)

    slot_is_positive = torch.arange(total_slots, device=sim.device) < num_positives
    is_own_image = torch.eye(batch_size, dtype=torch.bool, device=sim.device)

    pos_mask = slot_is_positive.view(1, 1, total_slots) & is_own_image.unsqueeze(-1)
    denom_mask = slot_is_positive.view(1, 1, total_slots) | is_own_image.unsqueeze(-1)
    if negative_valid is not None:
        # sim[i, b, s]: ảnh i so với slot s của ảnh b -> slot_valid[b, s] chọn theo chiều thứ 2
        slot_valid = torch.ones(batch_size, total_slots, dtype=torch.bool, device=sim.device)
        slot_valid[:, num_positives:] = negative_valid.to(device=sim.device, dtype=torch.bool)
        denom_mask = denom_mask & slot_valid.unsqueeze(0)

    sim_flat = sim.view(batch_size, batch_size * total_slots)
    pos_mask_flat = pos_mask.reshape(batch_size, batch_size * total_slots)
    denom_mask_flat = denom_mask.reshape(batch_size, batch_size * total_slots)

    denom = torch.logsumexp(sim_flat.masked_fill(~denom_mask_flat, float("-inf")), dim=1)
    pos_sim_sum = (sim_flat * pos_mask_flat).sum(dim=1)
    pos_count = pos_mask_flat.sum(dim=1).clamp_min(1)
    mean_pos_sim = pos_sim_sum / pos_count

    loss_per_anchor = denom - mean_pos_sim
    return loss_per_anchor.mean()


def _intercept_epoch_scale(epoch_index):
    if TRAIN_INTERCEPT_WARMUP_EPOCHS <= 0:
        return 1.0
    return min(1.0, float(epoch_index + 1) / float(TRAIN_INTERCEPT_WARMUP_EPOCHS))


def apply_decoder_object_intercept(
    outputs,
    image_paths,
    gt_objects_batch,
    object_token_ids,
    targets=None,
    epoch_index=0,
):
    """Dynamic logit adjustment cho token thuộc object vắng mặt.

    Khác bản static cũ, penalty không còn là một hằng số áp cho mọi timestep. Với mỗi ảnh:
    - xác định các object không có trong COCO instances,
    - đo tổng xác suất hiện tại mà model gán cho các token object vắng mặt ở từng timestep,
    - chỉ hạ logits mạnh khi xác suất nhóm này vượt ngưỡng,
    - bảo vệ token target đúng để CE không bị phạt ngược nếu annotation thiếu object.
    """
    if (
        not TRAIN_DECODER_INTERCEPT
        or TRAIN_INTERCEPT_MODE == "off"
        or not object_token_ids
    ):
        return outputs

    epoch_scale = _intercept_epoch_scale(epoch_index)
    if epoch_scale <= 0:
        return outputs

    # fp32 co dinh (khong ke thua dtype cua outputs) - duoi AMP, outputs la Half nhung
    # step_penalty tinh tu probs=outputs.softmax(...) lai la Float32 (autocast giu softmax
    # o fp32 de on dinh so hoc) -> gan Half=Float truoc day crash "Index put requires the
    # source and destination dtypes match". Ep fp32 xuyen suot, chi cast lai dtype cua
    # outputs 1 lan duy nhat luc return.
    full_bias = torch.zeros_like(outputs, dtype=torch.float32)
    probs = outputs.detach().softmax(dim=-1)
    max_penalty = TRAIN_INTERCEPT_MAX_PENALTY * epoch_scale

    for batch_index, image_path in enumerate(image_paths):
        gt_text = gt_objects_batch[batch_index] if gt_objects_batch is not None else ""
        present_objects = set(gt_text.split("|")) if gt_text else set()
        if not present_objects:
            continue

        absent_ids = [
            token_id
            for obj_name, token_ids in object_token_ids.items()
            if obj_name not in present_objects
            for token_id in token_ids
        ]
        if not absent_ids:
            continue

        absent_ids_t = torch.tensor(sorted(set(absent_ids)), device=outputs.device, dtype=torch.long)

        if TRAIN_INTERCEPT_MODE == "static":
            step_penalty = torch.full((outputs.size(1),), max_penalty, device=outputs.device, dtype=torch.float32)
        else:
            absent_mass = probs[batch_index].index_select(dim=-1, index=absent_ids_t).sum(dim=-1)
            normalized = (
                (absent_mass - TRAIN_INTERCEPT_PROB_THRESHOLD)
                / max(1e-6, 1.0 - TRAIN_INTERCEPT_PROB_THRESHOLD)
            ).clamp(min=0.0, max=1.0)
            step_penalty = max_penalty * normalized

        full_bias[batch_index, :, absent_ids_t] = -step_penalty.unsqueeze(-1)

        if TRAIN_INTERCEPT_TARGET_PROTECT and targets is not None:
            valid = targets[batch_index] != PAD_IDX
            timesteps = torch.arange(outputs.size(1), device=outputs.device)[valid]
            target_ids = targets[batch_index, valid]
            full_bias[batch_index, timesteps, target_ids] = 0.0

    return outputs + full_bias.to(outputs.dtype)

class CLIPEncoder(nn.Module):
    def __init__(self, embed_dim):
        super(CLIPEncoder, self).__init__()
        self.clip_vision = CLIPVisionModel.from_pretrained(
            "openai/clip-vit-base-patch16",
            use_safetensors=True,
        )

        for param in self.clip_vision.parameters():
            param.requires_grad = False

        self.fc = nn.Linear(768, embed_dim)
        self.layer_norm = nn.LayerNorm(embed_dim)

    def forward(self, images):
        outputs = self.clip_vision(pixel_values=images)
        features = outputs.last_hidden_state
        embeddings = self.fc(features)
        return self.layer_norm(embeddings)

class PositionalEncoding(nn.Module):
    def __init__(self, embed_dim, dropout=0.1, max_len=100):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, embed_dim)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, embed_dim, 2).float() * (-math.log(10000.0) / embed_dim))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        self.dropout = nn.Dropout(p=dropout)

    def forward(self, x):
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

class AttentionPool(nn.Module):
    """I006 diem 2 (Nhom B): thay cho `hidden_states.mean(dim=1)`. Mot query vector hoc duoc,
    attend qua toan bo chuoi token/patch roi weighted-sum - giu lai thong tin fine-grained
    thay vi coi moi vi tri quan trong nhu nhau (mean-pool cu). Dung chung 1 lop cho ca anh
    (patch tu CLIP-ViT, khong can key_padding_mask - moi patch deu hop le) va caption (decoder
    hidden state, can key_padding_mask de bo qua vi tri <pad>, giong tgt_key_padding_mask).

    Tham khao: `Triển khai I006_ 4 góp ý giảng viên cho Contrastive Loss.html` muc "Diem 2"."""
    def __init__(self, dim, num_heads=CONTRASTIVE_ATTN_POOL_HEADS):
        super().__init__()
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)
        self.attn = nn.MultiheadAttention(dim, num_heads=num_heads, batch_first=True)

    def forward(self, sequence, key_padding_mask=None):
        batch_size = sequence.size(0)
        q = self.query.expand(batch_size, -1, -1)
        pooled, _ = self.attn(q, sequence, sequence, key_padding_mask=key_padding_mask)
        return pooled.squeeze(1)  # (B, dim)


class ImageCaptioningTransformer(nn.Module):
    def __init__(self, vocab_size, embed_dim, ff_dim, num_heads, num_layers, contrastive_proj_dim=CONTRASTIVE_PROJ_DIM):
        super(ImageCaptioningTransformer, self).__init__()
        self.encoder = CLIPEncoder(embed_dim)
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=PAD_IDX)
        self.pos_encoder = PositionalEncoding(embed_dim)

        decoder_layer = nn.TransformerDecoderLayer(d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, batch_first=True)
        self.transformer_decoder = nn.TransformerDecoder(decoder_layer, num_layers=num_layers)
        self.fc_out = nn.Linear(embed_dim, vocab_size, bias=False)
        self.fc_out.weight = self.embedding.weight

        # --- Projection head cho contrastive loss (mới so với N006) ---
        # Tham số MỚI, không có trong checkpoint cũ -> load_checkpoint() phải dùng strict=False.
        self.image_proj = nn.Linear(embed_dim, contrastive_proj_dim)
        self.text_proj = nn.Linear(embed_dim, contrastive_proj_dim)
        # I006 diem 2: AttentionPool luon duoc khoi tao (checkpoint co the load lai du
        # POOLING_MODE la gi), chi KHONG duoc goi toi trong project_image/project_text khi
        # POOLING_MODE == "mean" (xem 2 method o duoi).
        self.image_pool = AttentionPool(embed_dim)
        self.text_pool = AttentionPool(embed_dim)

    def generate_square_subsequent_mask(self, sz):
        mask = (torch.triu(torch.ones(sz, sz)) == 1).transpose(0, 1)
        mask = mask.float().masked_fill(mask == 0, float('-inf')).masked_fill(mask == 1, float(0.0))
        return mask

    def encode_image(self, images):
        """Tách riêng bước encode ảnh để có thể dùng chung 1 lần `memory` cho cả forward pass
        positive VÀ negative caption trong contrastive loss (tránh forward lại CLIP-ViT 2 lần)."""
        return self.encoder(images)

    def decode(self, memory, captions, return_hidden=False):
        """Tách riêng bước decode để contrastive loss lấy được decoder hidden state (trước
        fc_out) làm text representation, không chỉ logits."""
        tgt_embed = self.embedding(captions)
        tgt_embed = self.pos_encoder(tgt_embed)
        tgt_mask = self.generate_square_subsequent_mask(captions.size(1)).to(captions.device)
        tgt_key_padding_mask = (captions == PAD_IDX)
        hidden = self.transformer_decoder(tgt=tgt_embed, memory=memory,
                                          tgt_mask=tgt_mask,
                                          tgt_key_padding_mask=tgt_key_padding_mask)
        logits = self.fc_out(hidden)
        if return_hidden:
            return logits, hidden
        return logits

    def forward(self, images, captions, neg_captions=None, return_hidden=False):
        """neg_captions=None: hanh vi cu (nhanh khong-contrastive / generate).
        neg_captions co gia tri, shape (B, N, L): encode anh 1 lan roi decode CA positive va
        TOAN BO N negative trong CUNG 1 lan forward() nay (memory duoc lap lai N lan theo batch
        truoc khi decode), de nn.DataParallel scatter/gather dung tren nhieu GPU (truoc day code
        goi thang model_.encode_image()/model_.decode() rieng, bo qua DataParallel.forward hoan
        toan -> chi chay tren 1 GPU). N=1 cho dung ket qua nhu truoc khi them multi-negative."""
        memory = self.encode_image(images)
        outputs, hidden = self.decode(memory, captions, return_hidden=True)
        if neg_captions is None:
            if return_hidden:
                return outputs, hidden
            return outputs

        batch_size, num_negatives, neg_len = neg_captions.shape
        neg_flat = neg_captions.reshape(batch_size * num_negatives, neg_len)
        memory_flat = memory.unsqueeze(1).expand(-1, num_negatives, -1, -1).reshape(
            batch_size * num_negatives, memory.size(1), memory.size(2)
        )
        _, neg_hidden_flat = self.decode(memory_flat, neg_flat, return_hidden=True)
        neg_text_embed = self.project_text(neg_hidden_flat, neg_flat == PAD_IDX).reshape(
            batch_size, num_negatives, -1
        )
        image_embed = self.project_image(memory)
        pos_text_embed = self.project_text(hidden, captions == PAD_IDX)
        return outputs, hidden, image_embed, pos_text_embed, neg_text_embed

    def project_image(self, memory):
        """v = normalize(W_v * pool(image_features)). I006 diem 2: pool = AttentionPool khi
        POOLING_MODE="attention" (moi), hoac mean(dim=1) khi POOLING_MODE="mean" (nhu N010)."""
        if POOLING_MODE == "attention":
            pooled = self.image_pool(memory)
        else:
            pooled = memory.mean(dim=1)
        return nn.functional.normalize(self.image_proj(pooled), dim=-1)

    def project_text(self, hidden, padding_mask):
        """t = normalize(W_t * pool(decoder_hidden_states)). I006 diem 2: pool = AttentionPool
        khi POOLING_MODE="attention" (key_padding_mask=padding_mask, dung nghia True=bo qua
        vi tri <pad>, giong tgt_key_padding_mask), hoac mean-pool tren vi tri KHONG phai pad
        khi POOLING_MODE="mean" (nhu N010)."""
        if POOLING_MODE == "attention":
            pooled = self.text_pool(hidden, key_padding_mask=padding_mask)
        else:
            valid = (~padding_mask).unsqueeze(-1).to(hidden.dtype)
            pooled = (hidden * valid).sum(dim=1) / valid.sum(dim=1).clamp_min(1.0)
        return nn.functional.normalize(self.text_proj(pooled), dim=-1)

In [ ]:
def _unwrap_model(model):
    return model.module if isinstance(model, nn.DataParallel) else model

def _prepare_save_path(filepath):
    """Đảm bảo checkpoint được lưu vào thư mục ghi được, không phải /kaggle/input."""
    if os.path.isdir(filepath):
        raise RuntimeError(
            f"'{filepath}' là thư mục, không phải file checkpoint. "
            "Hãy truyền CHECKPOINT_FILE hoặc BEST_MODEL_FILE, ví dụ /kaggle/working/last_checkpoint.pth."
        )
    save_dir = os.path.dirname(filepath) or "."
    os.makedirs(save_dir, exist_ok=True)
    if not os.access(save_dir, os.W_OK):
        raise RuntimeError(
            f"Không thể ghi checkpoint vào '{save_dir}'. "
            "Trên Kaggle, hãy lưu vào /kaggle/working và chỉ load checkpoint từ /kaggle/input."
        )
    return filepath

def _resolve_checkpoint_path(filepath):
    """Cho phép RESUME_CHECKPOINT_FILE là file .pth hoặc folder chứa checkpoint."""
    if os.path.isfile(filepath):
        return filepath

    if os.path.isdir(filepath):
        preferred_names = ["last_checkpoint.pth", "best_model.pth"]
        for name in preferred_names:
            candidate = os.path.join(filepath, name)
            if os.path.isfile(candidate):
                return candidate

        checkpoint_files = []
        for root, _, files in os.walk(filepath):
            for name in files:
                if name.endswith((".pth", ".pt")):
                    checkpoint_files.append(os.path.join(root, name))

        if checkpoint_files:
            checkpoint_files.sort(key=os.path.getmtime, reverse=True)
            return checkpoint_files[0]

        raise RuntimeError(f"'{filepath}' là thư mục nhưng không tìm thấy file .pth/.pt bên trong.")

    return filepath

class EarlyStoppingCallback:
    """So với bản fine-tune: hỗ trợ mode="max" để early-stop/chọn best model theo BLEU-4
    (điểm càng cao càng tốt) thay vì chỉ mode="min" theo val_loss.

    patience mặc định 5 (thay vì 3): BLEU-4 tính bằng greedy decode trên subset mỗi epoch nhiễu
    hơn nhiều so với val_loss (quá trình rời rạc, không mượt/đơn điệu như loss) - patience thấp dễ
    dừng sớm giữa 1 đợt dao động tạm thời, mất cơ hội cải thiện tiếp."""
    def __init__(self, patience=5, min_delta=0.0, save_path=BEST_MODEL_FILE, mode="min"):
        assert mode in ("min", "max")
        self.patience = patience
        self.min_delta = min_delta
        self.save_path = save_path
        self.mode = mode
        self.counter = 0
        self.best_score = float('inf') if mode == "min" else float('-inf')
        self.early_stop = False

    def _improved(self, score):
        if self.mode == "min":
            return score < self.best_score - self.min_delta
        return score > self.best_score + self.min_delta

    def __call__(self, score, model, optimizer, epoch, scheduler=None, scaler=None):
        if self._improved(score):
            self.best_score = score
            self.counter = 0
            self.save_best_model(model, optimizer, epoch, score, scheduler, scaler)
        else:
            self.counter += 1
            print(f" EarlyStopping counter: {self.counter} / {self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_best_model(self, model, optimizer, epoch, score, scheduler=None, scaler=None):
        checkpoint = {
            'epoch': epoch,
            'model_state_dict': _unwrap_model(model).state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            # Thiếu 2 trạng thái này thì resume từ best model sẽ warmup lại LR từ đầu.
            'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
            'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
            'seed': SEED,
            'monitor_metric': EARLY_STOP_METRIC,
            'best_score': self.best_score,
        }
        save_path = _prepare_save_path(self.save_path)
        torch.save(checkpoint, save_path)
        print(f" => {EARLY_STOP_METRIC} cải thiện ({score:.4f}). Đã lưu BEST MODEL tại: {save_path}")

# ==========================================
# QUẢN LÝ CHECKPOINT VÀ HUẤN LUYỆN
# ==========================================
def save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics, early_stopping, filepath=CHECKPOINT_FILE, scaler=None):
    """Lưu Checkpoint bao gồm cả trạng thái Early Stopping và toàn bộ metric của epoch (để vẽ lại sau)."""
    checkpoint = {
        'epoch': epoch,
        'model_state_dict': _unwrap_model(model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict() if scheduler is not None else None,
        'scaler_state_dict': scaler.state_dict() if scaler is not None else None,
        'seed': SEED,
        'epoch_metrics': epoch_metrics,
        'monitor_metric': EARLY_STOP_METRIC,
        'best_score': early_stopping.best_score,
        'es_counter': early_stopping.counter,
    }
    filepath = _prepare_save_path(filepath)
    torch.save(checkpoint, filepath)
    print(f" => Đã lưu Checkpoint phục hồi tại: {filepath}")

def load_checkpoint(model, optimizer=None, scheduler=None, early_stopping=None, filepath=CHECKPOINT_FILE, resume_training=True, reset_training_state=False, scaler=None):
    """Tải checkpoint; khi fine-tune có thể chỉ load trọng số và reset optimizer/scheduler/epoch."""
    start_epoch = 0
    if not resume_training:
        print("[*] RESUME_TRAINING = False. Sẽ bắt đầu huấn luyện từ Epoch 1.")
        return start_epoch

    filepath = _resolve_checkpoint_path(filepath)

    if os.path.exists(filepath):
        print(f"[*] Tìm thấy Checkpoint '{filepath}'. Đang tải...")
        checkpoint = torch.load(filepath, map_location=device)
        state_dict = checkpoint.get('model_state_dict', checkpoint)
        # strict=False: checkpoint cũ (vd bản không có contrastive loss) không có
        # image_proj/text_proj -> vẫn load được phần trọng số trùng, phần mới khởi tạo lại.
        load_result = _unwrap_model(model).load_state_dict(state_dict, strict=False)
        if load_result.missing_keys or load_result.unexpected_keys:
            print(f"[*] load_state_dict (strict=False): missing_keys={load_result.missing_keys}, "
                  f"unexpected_keys={load_result.unexpected_keys}")

        if reset_training_state:
            start_epoch = 0
            print("[*] Fine-tune mode: chỉ load trọng số model, reset optimizer/scheduler/epoch.")
        else:
            if optimizer is not None and 'optimizer_state_dict' in checkpoint:
                optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
            if scheduler is not None and checkpoint.get('scheduler_state_dict') is not None:
                scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
            if scaler is not None and checkpoint.get('scaler_state_dict') is not None:
                scaler.load_state_dict(checkpoint['scaler_state_dict'])
            start_epoch = checkpoint.get('epoch', -1) + 1
            if early_stopping is not None:
                early_stopping.best_score = checkpoint.get('best_score', early_stopping.best_score)
                early_stopping.counter = checkpoint.get('es_counter', 0)

        print(f"[*] Đã khôi phục thành công. Sẽ tiếp tục huấn luyện từ Epoch {start_epoch + 1}")
        if early_stopping is not None:
            print(f"[*] Trạng thái EarlyStopping ({early_stopping.mode}): best_score = {early_stopping.best_score:.4f}, counter = {early_stopping.counter}")
    else:
        print(f"[*] Không tìm thấy Checkpoint '{filepath}'. Sẽ bắt đầu huấn luyện từ Epoch 1.")
    return start_epoch

# ==========================================
# THEO DÕI CHẤT LƯỢNG CAPTION THẬT (BLEU-4, pycocoevalcap) MỖI EPOCH
# ==========================================
def build_val_reference_map(image_paths, captions, max_images=None):
    """Gộp các caption (đã standardize) của cùng 1 ảnh thành list reference string, đúng định
    dạng `gts` mà pycocoevalcap.bleu.Bleu.compute_score() cần (không tokenize thủ công như nltk)."""
    ref_map = {}
    order = []
    for path, cap in zip(image_paths, captions):
        if path not in ref_map:
            ref_map[path] = []
            order.append(path)
        ref_map[path].append(cap)
    if max_images is not None:
        order = order[:max_images]
    return {path: ref_map[path] for path in order}

@torch.no_grad()
def greedy_generate_tokens(model, image_tensor, max_length=SEQ_LENGTH):
    """Sinh caption bằng greedy decoding (nhanh) - CHỈ dùng nội bộ để theo dõi BLEU-4 mỗi epoch
    trong lúc train. Không phải hàm inference cuối cùng - notebook này không có bước generate/
    evaluate caption (xem DECODER_INTERCEPT.md cho constrained decoding lúc inference)."""
    model_ = _unwrap_model(model)
    was_training = model_.training
    model_.eval()

    image_tensor = image_tensor.unsqueeze(0).to(device)
    memory = model_.encoder(image_tensor)
    generated = [START_IDX]

    for _ in range(max_length - 1):
        tgt = torch.tensor(generated, dtype=torch.long, device=device).unsqueeze(0)
        tgt_mask = model_.generate_square_subsequent_mask(tgt.size(1)).to(device)
        tgt_embed = model_.pos_encoder(model_.embedding(tgt))
        out = model_.transformer_decoder(tgt=tgt_embed, memory=memory, tgt_mask=tgt_mask)
        next_token = model_.fc_out(out)[0, -1, :].argmax(dim=-1).item()
        if next_token == END_IDX:
            break
        generated.append(next_token)

    if was_training:
        model_.train()
    return generated[1:]  # bỏ <start>

def evaluate_val_bleu4(model, vocab, ref_map, max_images=VAL_METRIC_MAX_SAMPLES, transforms=None):
    """Tính BLEU-4 bằng pycocoevalcap (chuẩn dùng để báo cáo trong các paper/benchmark image
    captioning trên COCO) trên 1 subset ảnh val bằng greedy decode - KHÔNG dùng nltk vì nltk
    BLEU vốn thiết kế cho đánh giá dịch máy, không phải công cụ chuẩn cho captioning.
    Dùng để theo dõi/early-stop theo chất lượng caption thật thay vì chỉ nhìn val_loss.
    Bỏ qua bước PTBTokenizer (cần Java) vì caption đã được `custom_standardization` chuẩn hoá
    sẵn (viết thường, bỏ dấu câu) - giống cách evaluate_baseline.py trong repo đang dùng."""
    from pycocoevalcap.bleu.bleu import Bleu

    if transforms is None:
        transforms = val_transforms

    idx2word = {idx: word for word, idx in vocab.items()}
    gts, res = {}, {}

    for img_idx, image_path in enumerate(list(ref_map.keys())[:max_images]):
        try:
            image = Image.open(image_path).convert("RGB")
        except Exception:
            continue
        image_tensor = transforms(image)
        token_ids = greedy_generate_tokens(model, image_tensor)
        hyp_caption = " ".join(idx2word.get(tok, "<unk>") for tok in token_ids).strip()

        key = str(img_idx)
        gts[key] = ref_map[image_path]
        res[key] = [hyp_caption if hyp_caption else "<unk>"]

    if not res:
        return 0.0

    bleu_score, _ = Bleu(4).compute_score(gts, res)
    return bleu_score[3]  # BLEU-4

def train_model(train_loader, val_loader, vocab, object_token_ids=None, val_ref_map=None, id_to_obj_name=None):
    vocab_size = len(vocab)
    model = ImageCaptioningTransformer(
        vocab_size=vocab_size, embed_dim=EMBED_DIM,
        ff_dim=FF_DIM, num_heads=NUM_HEADS, num_layers=NUM_LAYERS,
        contrastive_proj_dim=CONTRASTIVE_PROJ_DIM
    ).to(device)

    if torch.cuda.is_available() and torch.cuda.device_count() > 1:
        print(f"[*] Dùng {torch.cuda.device_count()} GPU với nn.DataParallel")
        model = nn.DataParallel(model)

    # Label smoothing: giảm overconfidence của model (một nguyên nhân gây hallucination).
    criterion = nn.CrossEntropyLoss(ignore_index=PAD_IDX, label_smoothing=LABEL_SMOOTHING)
    optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=LEARNING_RATE, weight_decay=0.01)

    num_training_steps = len(train_loader) * EPOCHS
    num_warmup_steps = int(0.05 * num_training_steps)
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=num_warmup_steps,
        num_training_steps=num_training_steps
    )
    amp_enabled = USE_AMP and torch.cuda.is_available()
    scaler = torch.cuda.amp.GradScaler(enabled=amp_enabled)
    print(f"[*] Mixed precision (AMP): {'BAT' if amp_enabled else 'TAT'} (USE_AMP={USE_AMP}, cuda={torch.cuda.is_available()})")

    early_stop_mode = "min" if EARLY_STOP_METRIC == "loss" else "max"
    early_stopping = EarlyStoppingCallback(patience=5, save_path=BEST_MODEL_FILE, mode=early_stop_mode)

    # -------------------------
    # RESUME TỪ CHECKPOINT (chỉ để tiếp tục sau khi Kaggle kernel bị ngắt, không load baseline)
    # -------------------------
    start_epoch = load_checkpoint(
        model,
        optimizer,
        scheduler,
        early_stopping,
        filepath=RESUME_CHECKPOINT_FILE,
        resume_training=RESUME_TRAINING,
        reset_training_state=FINE_TUNE_RESET_TRAINING_STATE,
        scaler=scaler
    )

    # Checkpoint không có scheduler_state_dict (vd best model bản cũ) -> scheduler mới đứng ở bước 0 và
    # warmup lại LR lên đỉnh, làm model đã hội tụ bị đẩy văng. Tua scheduler tới đúng số bước đã train
    # (lấy từ state AdamW), hoặc dựng lịch mới với RESUME_PEAK_LR khi lịch cũ đã chạy hết.
    if start_epoch > 0 and RESUME_TRAINING and not FINE_TUNE_RESET_TRAINING_STATE:
        steps_per_epoch = len(train_loader)
        checkpoint_lr = optimizer.param_groups[0]['lr']
        adam_state = next(iter(optimizer.state.values()), {})
        done_steps = int(adam_state['step']) if 'step' in adam_state else start_epoch * steps_per_epoch
        if RESUME_PEAK_LR is not None:
            for group in optimizer.param_groups:
                group['lr'] = RESUME_PEAK_LR
                group['initial_lr'] = RESUME_PEAK_LR
            remaining_steps = steps_per_epoch * max(1, EPOCHS - start_epoch)
            scheduler = get_linear_schedule_with_warmup(
                optimizer,
                num_warmup_steps=int(0.05 * remaining_steps),
                num_training_steps=remaining_steps
            )
            print(f"[*] Resume với lịch LR mới: đỉnh {RESUME_PEAK_LR:g}, {remaining_steps} bước cho {EPOCHS - start_epoch} epoch còn lại.")
        elif scheduler.last_epoch < done_steps:
            scheduler.last_epoch = done_steps - 1
            scheduler.step()
            print(f"[*] Checkpoint không có trạng thái scheduler -> tua scheduler tới bước {done_steps}.")
        resume_lr = optimizer.param_groups[0]['lr']
        print(f"[*] LR trong checkpoint: {checkpoint_lr:.2e} | LR khi tiếp tục: {resume_lr:.2e}")
        if RESUME_PEAK_LR is None and checkpoint_lr < 0.01 * LEARNING_RATE and resume_lr > 0.1 * LEARNING_RATE:
            print("[!] CẢNH BÁO: lịch LR của checkpoint đã chạy hết (LR ~ 0) mà LR khi tiếp tục lại lớn. "
                  "Nhảy LR đột ngột dễ làm loss tăng hoặc phân kỳ; nên đặt RESUME_PEAK_LR (vd 2e-5).")

    log_interval = 200  # Bộ dữ liệu lớn, in thưa hơn để tránh trôi màn hình

    for epoch in range(start_epoch, EPOCHS):
        # Seed theo epoch giúp resume từ checkpoint vẫn tái lập đúng thứ tự batch/augmentation.
        epoch_seed = SEED + epoch
        seed_everything(epoch_seed)
        if train_loader.generator is not None:
            train_loader.generator.manual_seed(epoch_seed)
        print(f"\n--- Bắt đầu Epoch {epoch+1:02d}/{EPOCHS} ---")
        _epoch_t0 = __import__("time").time()
        # USE_HALL_LOSS=False -> lambda=0 nên hall_loss vẫn được TÍNH (để log/theo dõi qua
        # object_token_ids dùng chung với Dynamic Intercept/Contrastive) nhưng KHÔNG cộng vào loss.
        current_lambda_hall = get_hall_lambda(epoch, LAMBDA_HALL, HALL_LAMBDA_WARMUP_EPOCHS) if USE_HALL_LOSS else 0.0
        hall_min_step_index = int(HALL_LOSS_MIN_STEP_RATIO * SEQ_LENGTH)
        print(f"   [Hall Loss] lambda={current_lambda_hall:.5f} (max={LAMBDA_HALL}) | min_step_index={hall_min_step_index}")

        # -------------------------
        # PHASE 1: TRAINING
        # -------------------------
        current_lambda_con = get_contrastive_lambda(epoch, LAMBDA_CONTRASTIVE, CONTRASTIVE_WARMUP_EPOCHS) if USE_CONTRASTIVE_LOSS else 0.0
        if USE_CONTRASTIVE_LOSS:
            print(f"   [Contrastive] lambda={current_lambda_con:.5f} (max={LAMBDA_CONTRASTIVE}) | tau={CONTRASTIVE_TAU}")

        model.train()
        train_loss = 0
        train_ce_loss = 0
        train_hall_loss = 0
        train_con_loss = 0
        neg_valid_count, neg_total_count = 0, 0
        train_correct = 0
        train_tokens = 0

        for batch_idx, (images, captions, image_paths, gt_objects_batch, extra_pos_captions) in enumerate(train_loader):
            images, captions = images.to(device), captions.to(device)
            extra_pos_captions = extra_pos_captions.to(device)

            decoder_input = captions[:, :-1]
            targets = captions[:, 1:]

            optimizer.zero_grad()

            with torch.cuda.amp.autocast(enabled=amp_enabled):
                if USE_CONTRASTIVE_LOSS:
                    neg_captions_full, neg_valid = build_negative_captions(
                        captions, gt_objects_batch, object_token_ids, id_to_obj_name,
                        num_negatives=NUM_NEGATIVES, num_structural=NUM_NEGATIVES_STRUCTURAL,
                        object_freq_weights=object_freq_weights, return_valid=True,
                    )
                    # I006 diem 1+4: ghep (K_POS_PER_IMAGE - 1) caption THAT cua CUNG anh (extra
                    # positive, chi dung cho contrastive) voi negative tong hop, decode CA HAI
                    # trong CUNG 1 lan goi model() - tai su dung co che decode-N-cai da co san
                    # (tuong thich DataParallel), khong can sua forward().
                    extra_pos_decoder_input = extra_pos_captions[:, :, :-1]
                    neg_decoder_input = neg_captions_full[:, :, :-1]
                    combined_decoder_input = torch.cat([extra_pos_decoder_input, neg_decoder_input], dim=1)
                    # Goi qua `model` (nn.DataParallel neu >1 GPU), KHONG qua model_ unwrap -
                    # forward() gio encode anh 1 lan roi decode ca positive/negative TRONG CUNG
                    # 1 lan goi, nen DataParallel scatter/gather dung tren toan bo nhanh nay.
                    outputs, pos_hidden, image_embed, pos_text_embed, combined_text_embed = model(
                        images, decoder_input, neg_captions=combined_decoder_input, return_hidden=True
                    )
                else:
                    outputs = model(images, decoder_input)
                # Intercept vào decoder NGAY LÚC TRAIN: hạ logit của token thuộc object vắng mặt
                # trước khi tính CE/hall_loss, để model học trực tiếp việc bị chặn thay vì chỉ
                # bị phạt gián tiếp qua hall_loss.
                outputs = apply_decoder_object_intercept(
                    outputs, image_paths, gt_objects_batch, object_token_ids,
                    targets=targets, epoch_index=epoch
                )
                ce_loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
                if USE_HALL_LOSS:
                    hall_loss = hallucination_object_loss(
                        outputs, image_paths, gt_objects_batch, object_token_ids, token_mask=(targets != PAD_IDX)
                    )
                else:
                    hall_loss = outputs.new_tensor(0.0)

                if USE_CONTRASTIVE_LOSS:
                    # I006 diem 1+4 (SupCon - Khosla et al. 2020): tach lai extra-positive va
                    # negative tu combined_text_embed, ghep voi pos_text_embed (anchor CE) thanh
                    # TOAN BO positive that cua anh nay, roi tinh multi-positive InfoNCE - caption
                    # THAT cua anh khac trong batch tu dong la negative (in-batch, mien phi).
                    num_extra_pos = extra_pos_decoder_input.size(1)
                    extra_pos_text_embed = combined_text_embed[:, :num_extra_pos, :]
                    synth_neg_text_embed = combined_text_embed[:, num_extra_pos:, :]
                    all_pos_text_embed = torch.cat([pos_text_embed.unsqueeze(1), extra_pos_text_embed], dim=1)
                    all_text_embed = torch.cat([all_pos_text_embed, synth_neg_text_embed], dim=1)
                    con_loss = multi_positive_infonce_loss(
                        image_embed, all_text_embed, num_positives=1 + num_extra_pos, tau=CONTRASTIVE_TAU,
                        negative_valid=neg_valid,
                    )
                else:
                    con_loss = outputs.new_tensor(0.0)

                loss = ce_loss + (current_lambda_hall * hall_loss) + (current_lambda_con * con_loss)

            predictions = outputs.argmax(dim=-1)
            mask = (targets != PAD_IDX)
            correct = (predictions == targets) & mask

            train_correct += correct.sum().item()
            train_tokens += mask.sum().item()

            scaler.scale(loss).backward()
            # unscale truoc khi clip: clip phai tinh tren gradient that, khong phai gradient da scale boi AMP.
            scaler.unscale_(optimizer)
            # Gradient clipping: ổn định gradient khi train decoder từ đầu (không có baseline dẫn dắt).
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP_NORM)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            train_loss += loss.item()
            train_ce_loss += ce_loss.item()
            train_hall_loss += hall_loss.item()
            train_con_loss += con_loss.item()
            if USE_CONTRASTIVE_LOSS:
                neg_valid_count += int(neg_valid.sum().item())
                neg_total_count += neg_valid.numel()

            if (batch_idx + 1) % log_interval == 0 or (batch_idx + 1) == len(train_loader):
                batch_acc = (correct.sum().item() / mask.sum().item()) * 100 if mask.sum().item() > 0 else 0.0
                print(f"   [Epoch {epoch+1:02d}] Batch {batch_idx+1:05d}/{len(train_loader)} | Loss: {loss.item():.4f} | CE: {ce_loss.item():.4f} | Hall: {hall_loss.item():.6f} | Con: {con_loss.item():.6f} | Acc: {batch_acc:.2f}%")

        epoch_train_loss = train_loss / len(train_loader)
        epoch_train_ce_loss = train_ce_loss / len(train_loader)
        epoch_train_hall_loss = train_hall_loss / len(train_loader)
        epoch_train_con_loss = train_con_loss / len(train_loader)
        epoch_train_acc = (train_correct / train_tokens) * 100

        # -------------------------
        # PHASE 2: VALIDATION
        # -------------------------
        model.eval()
        val_loss = 0
        val_ce_loss = 0
        val_hall_loss = 0
        val_con_loss = 0
        val_correct = 0
        val_tokens = 0

        if val_loader.generator is not None:
            val_loader.generator.manual_seed(SEED + 100_000 + epoch)

        with torch.no_grad():
            for images, captions, image_paths, gt_objects_batch, extra_pos_captions in val_loader:
                images, captions = images.to(device), captions.to(device)
                extra_pos_captions = extra_pos_captions.to(device)

                decoder_input = captions[:, :-1]
                targets = captions[:, 1:]

                if USE_CONTRASTIVE_LOSS:
                    neg_captions_full, neg_valid = build_negative_captions(
                        captions, gt_objects_batch, object_token_ids, id_to_obj_name,
                        num_negatives=NUM_NEGATIVES, num_structural=NUM_NEGATIVES_STRUCTURAL,
                        object_freq_weights=object_freq_weights, return_valid=True,
                    )
                    extra_pos_decoder_input = extra_pos_captions[:, :, :-1]
                    neg_decoder_input = neg_captions_full[:, :, :-1]
                    combined_decoder_input = torch.cat([extra_pos_decoder_input, neg_decoder_input], dim=1)
                    outputs, pos_hidden, image_embed, pos_text_embed, combined_text_embed = model(
                        images, decoder_input, neg_captions=combined_decoder_input, return_hidden=True
                    )
                else:
                    outputs = model(images, decoder_input)
                # Giữ nhất quán với training: val_loss/val_acc phản ánh đúng phân phối "đã bị
                # chặn" mà model thực sự dùng lúc generate.
                outputs = apply_decoder_object_intercept(
                    outputs, image_paths, gt_objects_batch, object_token_ids,
                    targets=targets, epoch_index=epoch
                )
                ce_loss = criterion(outputs.reshape(-1, vocab_size), targets.reshape(-1))
                if USE_HALL_LOSS:
                    hall_loss = hallucination_object_loss(
                    outputs, image_paths, gt_objects_batch, object_token_ids, token_mask=(targets != PAD_IDX)
                )
                else:
                    hall_loss = outputs.new_tensor(0.0)

                if USE_CONTRASTIVE_LOSS:
                    num_extra_pos = extra_pos_decoder_input.size(1)
                    extra_pos_text_embed = combined_text_embed[:, :num_extra_pos, :]
                    synth_neg_text_embed = combined_text_embed[:, num_extra_pos:, :]
                    all_pos_text_embed = torch.cat([pos_text_embed.unsqueeze(1), extra_pos_text_embed], dim=1)
                    all_text_embed = torch.cat([all_pos_text_embed, synth_neg_text_embed], dim=1)
                    con_loss = multi_positive_infonce_loss(
                        image_embed, all_text_embed, num_positives=1 + num_extra_pos, tau=CONTRASTIVE_TAU,
                        negative_valid=neg_valid,
                    )
                else:
                    con_loss = outputs.new_tensor(0.0)

                loss = ce_loss + (current_lambda_hall * hall_loss) + (current_lambda_con * con_loss)
                val_loss += loss.item()
                val_ce_loss += ce_loss.item()
                val_hall_loss += hall_loss.item()
                val_con_loss += con_loss.item()

                predictions = outputs.argmax(dim=-1)
                mask = (targets != PAD_IDX)
                val_correct += ((predictions == targets) & mask).sum().item()
                val_tokens += mask.sum().item()

        epoch_val_loss = val_loss / len(val_loader)
        epoch_val_ce_loss = val_ce_loss / len(val_loader)
        epoch_val_hall_loss = val_hall_loss / len(val_loader)
        epoch_val_con_loss = val_con_loss / len(val_loader)
        epoch_val_acc = (val_correct / val_tokens) * 100

        # BLEU-4 thật trên subset val (greedy decode) - chỉ tính khi cần cho theo dõi/early-stop.
        epoch_val_bleu4 = None
        if EARLY_STOP_METRIC == "bleu4" and val_ref_map:
            epoch_val_bleu4 = evaluate_val_bleu4(model, vocab, val_ref_map, max_images=VAL_METRIC_MAX_SAMPLES)

        monitor_score = epoch_val_bleu4 if EARLY_STOP_METRIC == "bleu4" else epoch_val_loss

        # -------------------------
        # BÁO CÁO VÀ LƯU CHECKPOINT
        # -------------------------
        print(f"\n=> TỔNG KẾT EPOCH {epoch+1:02d}:")
        print(f"   Train Loss: {epoch_train_loss:.4f} | CE: {epoch_train_ce_loss:.4f} | Hall: {epoch_train_hall_loss:.6f} | Con: {epoch_train_con_loss:.6f} | Train Acc: {epoch_train_acc:.2f}%")
        if USE_CONTRASTIVE_LOSS and neg_total_count:
            print(f"   Negative dùng được (khác positive): {100 * neg_valid_count / neg_total_count:.1f}% ({neg_valid_count}/{neg_total_count} slot)")
        bleu_str = f" | BLEU-4: {epoch_val_bleu4:.4f}" if epoch_val_bleu4 is not None else ""
        print(f"   Val Loss  : {epoch_val_loss:.4f} | CE: {epoch_val_ce_loss:.4f} | Hall: {epoch_val_hall_loss:.6f} | Con: {epoch_val_con_loss:.6f} | Val Acc  : {epoch_val_acc:.2f}%{bleu_str}")

        # 1. Lưu Best Model theo EARLY_STOP_METRIC (mặc định BLEU-4, không phải val_loss)
        early_stopping(monitor_score, model, optimizer, epoch, scheduler, scaler)

        # 2. Luôn lưu Last Checkpoint sau mỗi epoch để Resume khi sập Kaggle kernel
        epoch_metrics = {
            "val_loss": epoch_val_loss,
            "val_ce_loss": epoch_val_ce_loss,
            "val_hall_loss": epoch_val_hall_loss,
            "val_con_loss": epoch_val_con_loss,
            "val_acc": epoch_val_acc,
            "val_bleu4": epoch_val_bleu4,
        }
        save_checkpoint(model, optimizer, scheduler, epoch, epoch_metrics, early_stopping, CHECKPOINT_FILE, scaler=scaler)

        # Dừng an toàn trước giới hạn phiên Kaggle (TRAIN_TIME_BUDGET_HOURS); checkpoint của epoch này đã lưu ở trên.
        if TRAIN_TIME_BUDGET_HOURS and epoch + 1 < EPOCHS:
            _now = __import__("time").time()
            _epoch_h = (_now - _epoch_t0) / 3600
            _elapsed_h = (_now - NOTEBOOK_START_TIME) / 3600
            if _elapsed_h + 1.1 * _epoch_h > TRAIN_TIME_BUDGET_HOURS:
                print(f"[Thời gian] Đã chạy {_elapsed_h:.2f} giờ, epoch vừa rồi {_epoch_h * 60:.0f} phút: không đủ cho "
                      f"epoch {epoch + 2} trong {TRAIN_TIME_BUDGET_HOURS} giờ. Dừng sau epoch {epoch + 1}; phiên sau đặt "
                      f"RESUME_TRAINING=True, RESUME_CHECKPOINT_FILE=<{os.path.basename(CHECKPOINT_FILE)} trong output>.")
                break

        if early_stopping.early_stop:
            print(f"⚠️ Dừng huấn luyện sớm tại Epoch {epoch+1} do {EARLY_STOP_METRIC} không cải thiện liên tiếp.")
            break

    return model, optimizer

In [ ]:
# ==========================================
# CHẨN ĐOÁN: FILENAME KARPATHY CÓ KHỚP VỚI COCO INSTANCES KHÔNG?
# ==========================================
def diagnose_hall_label_coverage(image_paths, image_objects, label="train"):
    """Cảnh báo sớm nguyên nhân PHỔ BIẾN NHẤT khiến hallucination_object_loss/
    apply_decoder_object_intercept không có tác dụng gì: HallDebug in ra `gt_images=0 |
    present_images=0` ở MỌI batch. Với Flickr30k, nhãn rút từ caption nên khớp gần như 100%; thấp nghĩa là parse caption lỗi,
    hoặc filename trong file instances không khớp filename ảnh thật (một số dataset Kaggle đổi
    tên file, bỏ tiền tố COCO_train2014_/COCO_val2014_)."""
    if not image_paths:
        return
    sample_basenames = [os.path.basename(p) for p in image_paths[:2000]]
    matched = sum(1 for name in sample_basenames if name in image_objects)
    coverage = matched / len(sample_basenames) * 100
    print(f"   [HallCheck-{label}] Khớp filename Karpathy <-> COCO instances: {matched}/{len(sample_basenames)} ({coverage:.1f}%) trên mẫu {len(sample_basenames)} ảnh đầu.")
    if coverage < 5:
        example_img = sample_basenames[0]
        example_ann = next(iter(image_objects.keys()), None)
        print(f"   [!] CẢNH BÁO: gần như không khớp filename -> hallucination_object_loss VÀ apply_decoder_object_intercept sẽ luôn = 0 (vô tác dụng), model đang train như baseline thường.")
        print(f"       Ví dụ tên ảnh Karpathy: '{example_img}'")
        print(f"       Ví dụ key trong COCO instances: '{example_ann}'")
        print(f"       -> Kiểm tra lại cách đọc caption (load_flickr_data) và build_caption_objects.")

# ==========================================
# THỰC THI (MAIN) - TRAIN FROM SCRATCH
# ==========================================
if __name__ == "__main__":
    print("0. Đang chuẩn bị Dataset và Vocabulary cho train-from-scratch Flickr30k...")
    print(f"   SEED={SEED} | cuDNN deterministic=True | benchmark=False")
    print(f"   FF_DIM={FF_DIM} | NUM_LAYERS={NUM_LAYERS} | LR={LEARNING_RATE} | EPOCHS={EPOCHS} | LABEL_SMOOTHING={LABEL_SMOOTHING}")
    print(f"   EARLY_STOP_METRIC={EARLY_STOP_METRIC} (VAL_METRIC_MAX_SAMPLES={VAL_METRIC_MAX_SAMPLES})")
    print(f"   TRAIN_DECODER_INTERCEPT={TRAIN_DECODER_INTERCEPT} | MODE={TRAIN_INTERCEPT_MODE} | MAX_PENALTY={TRAIN_INTERCEPT_MAX_PENALTY} | WARMUP={TRAIN_INTERCEPT_WARMUP_EPOCHS}")
    print(f"   USE_CONTRASTIVE_LOSS={USE_CONTRASTIVE_LOSS} | LAMBDA_CONTRASTIVE={LAMBDA_CONTRASTIVE} | TAU={CONTRASTIVE_TAU} | WARMUP={CONTRASTIVE_WARMUP_EPOCHS} | POOLING_MODE={POOLING_MODE}")

    # 0.1 Load danh sách ảnh và caption tập Train theo Karpathy split
    train_imgs, train_caps = load_flickr_data(TRAIN_SPLITS)
    print(f"   Nguồn split: {SPLIT_SOURCE} | thư mục ảnh: {FLICKR_IMG_DIR}")

    # 0.2 Load danh sách ảnh và caption tập Val theo Karpathy split
    val_imgs, val_caps = load_flickr_data("val")

    # 0.3 Xây dựng (hoặc Tải) Vocabulary từ tập Train và lưu ra file (đảm bảo file có thể load lại)
    vocab = build_or_load_vocab(train_caps, min_freq=MIN_WORD_FREQ, vocab_file=VOCAB_FILE)
    vocab_size = len(vocab)

    # USE_OBJECT_GROUNDING = USE_HALL_LOSS or TRAIN_DECODER_INTERCEPT or USE_CONTRASTIVE_LOSS:
    # dựng hạ tầng object/COCO-instances khi BẤT KỲ thành phần nào cần, không chỉ riêng HalluLoss
    # (Dynamic Decoder Intercept và Contrastive loss dùng chung dữ liệu này).
    object_token_ids = build_object_token_ids(vocab) if USE_OBJECT_GROUNDING else {}
    val_image_objects = build_caption_objects(val_imgs, val_caps) if (USE_OBJECT_GROUNDING and USE_CAPTION_OBJECT_LABELS) else {}
    train_image_objects = build_caption_objects(train_imgs, train_caps) if (USE_OBJECT_GROUNDING and USE_CAPTION_OBJECT_LABELS) else {}
    if USE_OBJECT_GROUNDING:
        print(f"   Object grounding: USE_HALL_LOSS={USE_HALL_LOSS} (lambda={LAMBDA_HALL}) | "
              f"TRAIN_DECODER_INTERCEPT={TRAIN_DECODER_INTERCEPT} | USE_CONTRASTIVE_LOSS={USE_CONTRASTIVE_LOSS} | "
              f"nhãn object rút từ 5 caption/ảnh (không dùng YOLO)={USE_CAPTION_OBJECT_LABELS}")
        if USE_CAPTION_OBJECT_LABELS:
            diagnose_hall_label_coverage(train_imgs, train_image_objects, label="train")
            diagnose_hall_label_coverage(val_imgs, val_image_objects, label="val")

    # Contrastive loss (research/contrastive-loss-for-caption-hallucination.md) tái dùng đúng
    # object_token_ids đã build ở trên (qua USE_OBJECT_GROUNDING) - cần object_token_ids không rỗng.
    id_to_obj_name = build_object_id_to_name(object_token_ids) if USE_CONTRASTIVE_LOSS else {}
    # Tần suất category THẬT trong ground-truth captions (train_caps) - dùng làm trọng số
    # chọn object hallucination thay vì random đều trên 80 category. Xem
    # scratchpad_audit_do_kho_negative.html (căn cứ: Rohrbach et al. 2018).
    object_freq_weights = build_object_frequency_weights(train_caps, object_token_ids) if USE_CONTRASTIVE_LOSS else {}
    if USE_CONTRASTIVE_LOSS:
        print(f"   Contrastive loss: lambda={LAMBDA_CONTRASTIVE}, tau={CONTRASTIVE_TAU}, "
              f"warmup_epochs={CONTRASTIVE_WARMUP_EPOCHS}, proj_dim={CONTRASTIVE_PROJ_DIM}, " f"pooling_mode={POOLING_MODE}")
        if not object_token_ids:
            print("   [!] CẢNH BÁO: object_token_ids rỗng -> build_negative_captions sẽ không có "
                  "object nào để tạo negative, contrastive loss sẽ luôn ~0 (vô tác dụng).")

    # 0.4 Kiểm tra pycocoevalcap trước khi train (fail-fast) - nếu thiếu thì fallback về "loss"
    # thay vì crash giữa chừng epoch đầu tiên khi evaluate_val_bleu4 được gọi.
    if EARLY_STOP_METRIC == "bleu4":
        try:
            from pycocoevalcap.bleu.bleu import Bleu  # noqa: F401
        except ImportError:
            print("[!] Không tìm thấy pycocoevalcap -> fallback EARLY_STOP_METRIC='loss'. "
                  "Cài `pip install pycocoevalcap` để dùng BLEU-4 chuẩn COCO caption cho early-stopping.")
            EARLY_STOP_METRIC = "loss"

    # 0.5 Reference map (nhiều caption / ảnh) để tính BLEU-4 thật mỗi epoch, nếu được bật.
    val_ref_map = None
    if EARLY_STOP_METRIC == "bleu4":
        val_ref_map = build_val_reference_map(val_imgs, val_caps, max_images=VAL_METRIC_MAX_SAMPLES)
        print(f"   -> Sẽ tính BLEU-4 mỗi epoch trên {len(val_ref_map)} ảnh val (greedy decode).")

    # 1. Khởi tạo Dataset & DataLoader
    _k_pos = K_POS_PER_IMAGE if USE_CONTRASTIVE_LOSS else 1  # I006 diem 1: chi lay extra
    # positive khi thuc su dung contrastive loss, tranh tokenize thua khi tat.
    train_dataset = FlickrDataset(train_imgs, train_caps, vocab, transforms=train_transforms, image_objects=train_image_objects, k_pos_per_image=_k_pos)
    val_dataset = FlickrDataset(val_imgs, val_caps, vocab, transforms=val_transforms, image_objects=val_image_objects, k_pos_per_image=_k_pos)

    print("1. Đang khởi tạo DataLoaders...")
    train_generator = torch.Generator()
    train_generator.manual_seed(SEED)
    val_generator = torch.Generator()
    val_generator.manual_seed(SEED + 1)
    train_loader = DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True,
        num_workers=4, persistent_workers=True, pin_memory=True if torch.cuda.is_available() else False,
        worker_init_fn=seed_worker, generator=train_generator
    )
    val_loader = DataLoader(
        val_dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=4, persistent_workers=True, pin_memory=True if torch.cuda.is_available() else False,
        worker_init_fn=seed_worker, generator=val_generator
    )

    print("2. Bắt đầu train CLIP-ViT-B/16 + Transformer TỪ ĐẦU với hallucination-aware loss trên Flickr30k...")
    model, optimizer = train_model(
        train_loader,
        val_loader,
        vocab=vocab,
        object_token_ids=object_token_ids,
        val_ref_map=val_ref_map,
        id_to_obj_name=id_to_obj_name,
    )

    print("3. Quá trình huấn luyện hoàn tất toàn bộ số epochs yêu cầu.")

## Đánh giá sau khi train (tập test Karpathy, 1.000 ảnh)

Cell kế tiếp nạp **best model** (`BEST_MODEL_FILE`), sinh câu cho tập test bằng greedy và beam search rồi tính:

- **Chất lượng câu**: BLEU-1..4, METEOR (cần Java), ROUGE-L, CIDEr; SPICE tùy chọn (`EVAL_USE_SPICE`, chậm, cần Java và tải Stanford CoreNLP).
- **Ảo giác đối tượng (CHAIRs / CHAIRi)**: dùng bảng đồng nghĩa CHAIR gốc (Rohrbach et al., 2018). Nhãn đúng lấy từ 5 caption tham chiếu của ảnh
  vì Flickr30k không có COCO instances, nên con số cao hơn và **không so trực tiếp được** với CHAIR trên COCO.

Kết quả lưu ở `EVAL_RESULT_FILE` (JSON). Nếu kernel đã khởi động lại, chạy lại các cell định nghĩa (cấu hình, dữ liệu, mô hình) rồi chạy cell này; cell tự nạp từ điển và checkpoint từ `/kaggle/working`.


In [ ]:
# ==========================================
# ĐÁNH GIÁ SAU KHI TRAIN: tập test Karpathy của Flickr30k
# ==========================================
import shutil
import time
import torch.nn.functional as F

EVAL_SPLIT = "test"
EVAL_CHECKPOINT = BEST_MODEL_FILE   # đổi thành CHECKPOINT_FILE để đánh giá epoch cuối
EVAL_BEAM_SIZE = 3
EVAL_LENGTH_ALPHA = 0.7             # chọn câu beam theo logprob / độ dài^alpha
EVAL_MAX_IMAGES = None              # None = toàn bộ; đặt số nhỏ (vd 50) để chạy thử nhanh
EVAL_BATCH_SIZE = 50
EVAL_USE_METEOR = True              # cần Java; thiếu Java thì tự bỏ qua
EVAL_USE_SPICE = False              # cần Java + tải Stanford CoreNLP, chậm (hàng chục phút với 1.000 ảnh)
EVAL_RESULT_FILE = os.path.join(OUTPUT_DIR, "flickr30k_karpathy_dynlogits_contrastive_attnpool_eval_test.json")

CHAIR_SYNONYMS_TEXT = """person, girl, boy, man, woman, kid, child, chef, baker, people, adult, rider, children, baby, worker, passenger, sister, biker, policeman, cop, officer, lady, cowboy, bride, groom, male, female, guy, traveler, mother, father, gentleman, pitcher, player, skier, snowboarder, skater, skateboarder, person, woman, guy, foreigner, child, gentleman, caller, offender, coworker, trespasser, patient, politician, soldier, grandchild, serviceman, walker, drinker, doctor, bicyclist, thief, buyer, teenager, student, camper, driver, solider, hunter, shopper, villager
bicycle, bike, bicycle, bike, unicycle, minibike, trike
car, automobile, van, minivan, sedan, suv, hatchback, cab, jeep, coupe, taxicab, limo, taxi
motorcycle, scooter,  motor bike, motor cycle, motorbike, scooter, moped
airplane, jetliner, plane, air plane, monoplane, aircraft, jet, jetliner, airbus, biplane, seaplane
bus, minibus, trolley
train, locomotive, tramway, caboose
truck, pickup, lorry, hauler, firetruck
boat, ship, liner, sailboat, motorboat, dinghy, powerboat, speedboat, canoe, skiff, yacht, kayak, catamaran, pontoon, houseboat, vessel, rowboat, trawler, ferryboat, watercraft, tugboat, schooner, barge, ferry, sailboard, paddleboat, lifeboat, freighter, steamboat, riverboat, battleship, steamship
traffic light, street light, traffic signal, stop light, streetlight, stoplight
fire hydrant, hydrant
stop sign
parking meter
bench, pew
bird, ostrich, owl, seagull, goose, duck, parakeet, falcon, robin, pelican, waterfowl, heron, hummingbird, mallard, finch, pigeon, sparrow, seabird, osprey, blackbird, fowl, shorebird, woodpecker, egret, chickadee, quail, bluebird, kingfisher, buzzard, willet, gull, swan, bluejay, flamingo, cormorant, parrot, loon, gosling, waterbird, pheasant, rooster, sandpiper, crow, raven, turkey, oriole, cowbird, warbler, magpie, peacock, cockatiel, lorikeet, puffin, vulture, condor, macaw, peafowl, cockatoo, songbird
cat, kitten, feline, tabby
dog, puppy, beagle, pup, chihuahua, schnauzer, dachshund, rottweiler, canine, pitbull, collie, pug, terrier, poodle, labrador, doggie, doberman, mutt, doggy, spaniel, bulldog, sheepdog, weimaraner, corgi, cocker, greyhound, retriever, brindle, hound, whippet, husky
horse, colt, pony, racehorse, stallion, equine, mare, foal, palomino, mustang, clydesdale, bronc, bronco
sheep, lamb, ram, lamb, goat, ewe
cow, cattle, oxen, ox, calf, cattle, holstein, heifer, buffalo, bull, zebu, bison
elephant
bear, panda
zebra
giraffe
backpack, knapsack
umbrella
handbag, wallet, purse, briefcase
tie, bow, bow tie
suitcase, suit case, luggage
frisbee
skis, ski
snowboard
sports ball, ball
kite
baseball bat
baseball glove
skateboard
surfboard, longboard, skimboard, shortboard, wakeboard
tennis racket, racket
bottle
wine glass
cup
fork
knife, pocketknife, knive
spoon
bowl, container
banana
apple
sandwich, burger, sub, cheeseburger, hamburger
orange
broccoli
carrot
hot dog
pizza
donut, doughnut, bagel
cake,  cheesecake, cupcake, shortcake, coffeecake, pancake
chair, seat, stool
couch, sofa, recliner, futon, loveseat, settee, chesterfield
potted plant, houseplant
bed
dining table, table, desk
toilet, urinal, commode, toilet, lavatory, potty
tv, monitor, televison, television
laptop, computer, notebook, netbook, lenovo, macbook, laptop computer
mouse
remote
keyboard
cell phone, mobile phone, phone, cellphone, telephone, phon, smartphone, iPhone
microwave
oven, stovetop, stove, stove top oven
toaster
sink
refrigerator, fridge, fridge, freezer
book
clock
vase
scissors
teddy bear, teddybear
hair drier, hairdryer
toothbrush"""
_CHAIR_DOUBLE_WORDS = [
    "motor bike", "motor cycle", "air plane", "traffic light", "street light", "traffic signal",
    "stop light", "fire hydrant", "stop sign", "parking meter", "suit case", "sports ball",
    "baseball bat", "baseball glove", "tennis racket", "wine glass", "hot dog", "cell phone",
    "mobile phone", "teddy bear", "hair drier", "potted plant", "bow tie", "laptop computer",
    "stove top oven", "home plate", "train track",
]


def _load_chair_tables():
    objects, inverse = set(), {}
    for line in CHAIR_SYNONYMS_TEXT.splitlines():
        syns = [s.strip() for s in line.split(",") if s.strip()]
        if not syns:
            continue
        objects.update(syns)
        for s in syns:
            inverse[s] = syns[0]
    double = {w: w for w in _CHAIR_DOUBLE_WORDS}
    for w in ["bird", "cat", "dog", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe", "animal", "cub"]:
        double[f"baby {w}"] = w
        double[f"adult {w}"] = w
    for w in ["jet", "train"]:
        double[f"passenger {w}"] = w
    double.update({"bow tie": "tie", "toilet seat": "toilet", "wine glas": "wine glass"})
    return objects, inverse, double


_CHAIR_OBJECTS, _CHAIR_INVERSE, _CHAIR_DOUBLE = _load_chair_tables()
try:
    from nltk.stem import WordNetLemmatizer
    _LEMMA = WordNetLemmatizer()
    _LEMMA.lemmatize("dogs", pos="n")
except Exception:
    try:
        import nltk
        nltk.download("wordnet", quiet=True)
        from nltk.stem import WordNetLemmatizer
        _LEMMA = WordNetLemmatizer()
        _LEMMA.lemmatize("dogs", pos="n")
    except Exception:
        _LEMMA = None  # không có wordnet: dùng luật số nhiều đơn giản bên dưới


def _singular(word):
    if _LEMMA is not None:
        return _LEMMA.lemmatize(word, pos="n")
    if word.endswith("ies") and len(word) > 4:
        return word[:-3] + "y"
    if word.endswith(("ses", "xes", "ches", "shes")):
        return word[:-2]
    if word.endswith("s") and not word.endswith("ss") and len(word) > 3:
        return word[:-1]
    return word


def chair_objects(caption):
    """Các lớp COCO (từ chuẩn) được caption nhắc tới, theo cách chair.py của CHAIR."""
    words = [_singular(w) for w in caption.lower().split()]
    merged, i = [], 0
    while i < len(words):
        two = " ".join(words[i:i + 2])
        if two in _CHAIR_DOUBLE:
            merged.append(_CHAIR_DOUBLE[two])
            i += 2
        else:
            merged.append(words[i])
            i += 1
    if "toilet" in merged and "seat" in merged:
        merged = [w for w in merged if w != "seat"]
    return [_CHAIR_INVERSE[w] for w in merged if w in _CHAIR_OBJECTS]


def compute_chair(preds, refs_list):
    """CHAIRs: % câu nhắc ít nhất 1 object không có trong 5 caption tham chiếu; CHAIRi: % lượt nhắc object bị ảo giác."""
    n_caps = n_hall_caps = n_mentions = n_hall = skipped = 0
    for pred, refs in zip(preds, refs_list):
        gt = set()
        for r in refs:
            gt.update(chair_objects(r))
        if not gt:
            skipped += 1
            continue
        mentioned = chair_objects(pred)
        bad = [o for o in mentioned if o not in gt]
        n_caps += 1
        n_mentions += len(mentioned)
        n_hall += len(bad)
        n_hall_caps += bool(bad)
    return {"CHAIRs": 100.0 * n_hall_caps / max(n_caps, 1), "CHAIRi": 100.0 * n_hall / max(n_mentions, 1),
            "num_captions": n_caps, "skipped_no_ground_truth": skipped}


def _ids_to_text(ids, idx2word):
    out = []
    for t in ids:
        if t == END_IDX:
            break
        if t in (START_IDX, PAD_IDX):
            continue
        out.append(idx2word.get(t, "<unk>"))
    return " ".join(out)


@torch.no_grad()
def decode_greedy(model_, images):
    memory = model_.encoder(images)
    B = images.size(0)
    seqs = torch.full((B, 1), START_IDX, dtype=torch.long, device=images.device)
    done = torch.zeros(B, dtype=torch.bool, device=images.device)
    for _ in range(SEQ_LENGTH - 1):
        mask = model_.generate_square_subsequent_mask(seqs.size(1)).to(images.device)
        out = model_.transformer_decoder(tgt=model_.pos_encoder(model_.embedding(seqs)), memory=memory, tgt_mask=mask)
        nxt = model_.fc_out(out[:, -1]).argmax(-1).masked_fill(done, PAD_IDX)
        seqs = torch.cat([seqs, nxt.unsqueeze(1)], dim=1)
        done = done | (nxt == END_IDX)
        if done.all():
            break
    return seqs.tolist()


@torch.no_grad()
def decode_beam(model_, images, beam=3, alpha=0.7):
    memory = model_.encoder(images)
    B, L, D = memory.shape
    mem = memory.unsqueeze(1).expand(-1, beam, -1, -1).reshape(B * beam, L, D)
    seqs = torch.full((B * beam, 1), START_IDX, dtype=torch.long, device=images.device)
    scores = torch.zeros(B, beam, device=images.device)
    scores[:, 1:] = -1e9   # bước đầu chỉ giữ 1 nhánh để các beam không trùng nhau
    done = torch.zeros(B * beam, dtype=torch.bool, device=images.device)
    base = (torch.arange(B, device=images.device) * beam).unsqueeze(1)
    for _ in range(SEQ_LENGTH - 1):
        mask = model_.generate_square_subsequent_mask(seqs.size(1)).to(images.device)
        out = model_.transformer_decoder(tgt=model_.pos_encoder(model_.embedding(seqs)), memory=mem, tgt_mask=mask)
        logp = F.log_softmax(model_.fc_out(out[:, -1]).float(), dim=-1)
        V = logp.size(-1)
        logp[done] = -1e9
        logp[done, PAD_IDX] = 0.0   # nhánh đã kết thúc chỉ được nối <pad>, không đổi điểm
        cand = (scores.reshape(-1, 1) + logp).reshape(B, beam * V)
        top_scores, top_idx = cand.topk(beam, dim=1)
        sel = (base + top_idx // V).reshape(-1)
        tok = (top_idx % V).reshape(-1)
        seqs = torch.cat([seqs[sel], tok.unsqueeze(1)], dim=1)
        done = done[sel] | (tok == END_IDX)
        scores = top_scores
        if done.all():
            break
    lengths = ((seqs != PAD_IDX) & (seqs != START_IDX)).sum(dim=1).clamp_min(1).float().reshape(B, beam)
    best = (scores / lengths.pow(alpha)).argmax(dim=1)
    return seqs.reshape(B, beam, -1)[torch.arange(B, device=images.device), best].tolist()


def compute_caption_metrics(preds, refs_list):
    """BLEU/METEOR/ROUGE-L/CIDEr(/SPICE) bằng pycocoevalcap trên chữ đã chuẩn hóa sẵn (không cần PTBTokenizer). Điểm nhân 100."""
    from pycocoevalcap.bleu.bleu import Bleu
    from pycocoevalcap.cider.cider import Cider
    from pycocoevalcap.rouge.rouge import Rouge
    gts = {i: refs for i, refs in enumerate(refs_list)}
    res = {i: [p if p else "<unk>"] for i, p in enumerate(preds)}
    metrics = {}
    bleu, _ = Bleu(4).compute_score(gts, res, verbose=0)
    for k in range(4):
        metrics[f"BLEU-{k + 1}"] = 100 * bleu[k]
    has_java = shutil.which("java") is not None
    if EVAL_USE_METEOR and has_java:
        try:
            from pycocoevalcap.meteor.meteor import Meteor
            metrics["METEOR"] = 100 * Meteor().compute_score(gts, res)[0]
        except Exception as exc:
            print(f"[!] Bỏ qua METEOR: {exc}")
    elif EVAL_USE_METEOR:
        print("[!] Không có Java trong PATH: bỏ qua METEOR.")
    metrics["ROUGE-L"] = 100 * Rouge().compute_score(gts, res)[0]
    metrics["CIDEr"] = 100 * Cider().compute_score(gts, res)[0]
    if EVAL_USE_SPICE and has_java:
        try:
            from pycocoevalcap.spice.spice import Spice
            metrics["SPICE"] = 100 * Spice().compute_score(gts, res)[0]
        except Exception as exc:
            print(f"[!] Bỏ qua SPICE: {exc}")
    return metrics


class _EvalDataset(Dataset):
    def __init__(self, paths, transforms):
        self.paths, self.transforms = paths, transforms

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
        except Exception:
            img = Image.new("RGB", IMAGE_SIZE, (0, 0, 0))
        return self.transforms(img)


def load_eval_model(checkpoint_file, vocab_size):
    m = ImageCaptioningTransformer(vocab_size=vocab_size, embed_dim=EMBED_DIM, ff_dim=FF_DIM, num_heads=NUM_HEADS,
                                   num_layers=NUM_LAYERS, contrastive_proj_dim=CONTRASTIVE_PROJ_DIM).to(device)
    path = _resolve_checkpoint_path(checkpoint_file)
    try:
        ckpt = torch.load(path, map_location=device, weights_only=False)
    except TypeError:
        ckpt = torch.load(path, map_location=device)
    m.load_state_dict(ckpt.get("model_state_dict", ckpt))
    m.eval()
    print(f"[*] Đã nạp {path} (epoch {ckpt.get('epoch', -1) + 1 if isinstance(ckpt, dict) else '?'})")
    return m


def run_evaluation():
    eval_vocab = json.load(open(VOCAB_FILE, "r", encoding="utf-8"))
    idx2word = {i: w for w, i in eval_vocab.items()}
    paths, caps = load_flickr_data(EVAL_SPLIT)
    refs_by_path, order = {}, []
    for p, c in zip(paths, caps):
        if p not in refs_by_path:
            refs_by_path[p] = []
            order.append(p)
        refs_by_path[p].append(c)
    if EVAL_MAX_IMAGES:
        order = order[:EVAL_MAX_IMAGES]
    refs_list = [refs_by_path[p] for p in order]
    print(f"[*] Đánh giá {len(order)} ảnh của split '{EVAL_SPLIT}' bằng {os.path.basename(EVAL_CHECKPOINT)}")

    model_ = load_eval_model(EVAL_CHECKPOINT, len(eval_vocab))
    loader = DataLoader(_EvalDataset(order, val_transforms), batch_size=EVAL_BATCH_SIZE, shuffle=False, num_workers=2)

    results, samples = {}, {}
    for name, fn in (("greedy", lambda x: decode_greedy(model_, x)),
                     (f"beam{EVAL_BEAM_SIZE}", lambda x: decode_beam(model_, x, EVAL_BEAM_SIZE, EVAL_LENGTH_ALPHA))):
        t0 = time.time()
        preds = []
        for images in loader:
            for ids in fn(images.to(device)):
                preds.append(_ids_to_text(ids, idx2word))
        decode_s = time.time() - t0
        metrics = compute_caption_metrics(preds, refs_list)
        metrics.update(compute_chair(preds, refs_list))
        metrics["mean_len"] = sum(len(p.split()) for p in preds) / max(len(preds), 1)
        metrics["unk_rate_%"] = 100 * sum(p.split().count("<unk>") for p in preds) / max(sum(len(p.split()) for p in preds), 1)
        metrics["decode_seconds"] = decode_s
        results[name] = {k: (round(v, 2) if isinstance(v, float) else v) for k, v in metrics.items()}
        samples[name] = [{"image": os.path.basename(p), "pred": c, "refs": r[:2]} for p, c, r in list(zip(order, preds, refs_list))[:10]]
        print(f"\n=== {name} ===")
        for k, v in results[name].items():
            print(f"   {k:>16}: {v}")

    out = {"checkpoint": EVAL_CHECKPOINT, "split": EVAL_SPLIT, "split_source": SPLIT_SOURCE, "num_images": len(order),
           "beam_size": EVAL_BEAM_SIZE, "results": results, "samples": samples}
    with open(EVAL_RESULT_FILE, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=1)
    print(f"\n[*] Đã lưu kết quả: {EVAL_RESULT_FILE}")
    return out


if __name__ == "__main__":
    eval_out = run_evaluation()
